# Conditional diffusion U-Net — final frozen evaluation

This notebook freezes the selected diffusion recipe and evaluates it without further tuning. It reuses the trained `d10a` checkpoint and scores one final sampler, labelled `d10f`, on validation and test.

Frozen recipe:

- checkpoint: `ddpm_v10a_best.pt`
- weights: raw checkpoint weights, not EMA
- sampler: DDIM, `steps=250`, `eta=0.50`, `x_T_scale=1.00`
- clipping: no per-step `x0` clipping; final raw burned-fraction cap at train `p99.9` (`raw_p999`)
- ensemble size: 50


## 1. Imports and configuration

Defaults are set for the full v10 training-objective run on the 32 × 54 grid. For a smoke test, set `FAST_DEV_RUN = True`; restore the defaults before producing validation metrics for model selection.


In [ ]:
from pathlib import Path
from contextlib import nullcontext
import json
import math
import os
import random
import time
import uuid
import warnings

import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings("default")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True

RANDOM_SEED = 42


def seed_everything(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(RANDOM_SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch version: {torch.__version__}")
print(f"Device: {DEVICE}")

# Project constants.
BURN_THRESHOLD = 1e-6
LAND_THRESHOLD = 0.5
EXTREME_Q = 0.99
FIRE_SEASON_MONTHS = (7, 8, 9, 10)
HEADLINE_SUBSETS = ("all", "fire_season_Jul-Oct", "active_burned_gt_1e-6", "top_1pct_train_tail")
SELECTED_DETERMINISTIC_RUN_ID = "mild_active2_tail2"
SELECTED_DETERMINISTIC_MODEL = f"unet_{SELECTED_DETERMINISTIC_RUN_ID}"

# Diffusion target space.
# v6: diffuse transformed burned fraction, not raw y mapped to [-1, 1].
# v5 at y_scale=1e-3 stabilized one-step reconstruction but still under/over-shot
# the sampler. v6 tries a gentler log transform at y_scale=1e-2.
TARGET_MODE = "log1p_zscore"  # alternatives retained in code: "minus1_to_plus1"
LOG1P_TARGET_Y_SCALE = 1e-2
CLAMP_FINAL_INVERSE_TO_BURN_RANGE = True

# v11 final frozen evaluation. The checkpoint was selected from validation-only
# training/sampler tuning: d10a = active/tail 2/2 + x0 auxiliary 0.10.
# This notebook does not train; it reuses ddpm_v10a_best.pt and evaluates the
# frozen sampler on validation and test.
D10_OBJECTIVE_CONFIG = {
    "experiment_id": "ddpm_v10a",
    "model_name": "d10a",
    "active_cell_loss_weight": 2.0,
    "tail_cell_loss_weight": 2.0,
    "x0_aux_loss_weight": 0.10,
    "description": "locked d10a: active/tail 2/2 + x0 auxiliary 0.10",
}
SOURCE_OVERFIT_EXPERIMENT_ID = "conditional_ddpm_unet_log1p_zscore_yscale1em2_x0aux005_v6"
PREVIOUS_FULL_EXPERIMENT_ID = "ddpm_v10a"
SOURCE_FULL_EXPERIMENT_ID = "ddpm_v10a"
DIFFUSION_EXPERIMENT_ID = "ddpm_v11_final_eval"
DIFFUSION_MODEL_NAME = "d10f"

RUN_OVERFIT_DEBUG = False
STOP_AFTER_OVERFIT_DEBUG = False
RUN_V7_SAMPLER_CALIBRATION = False

# Evaluation controls. This is the first test-set evaluation of the locked diffusion recipe.
RUN_TRAIN = False
RESUME_TRAINING = False
RUN_VAL_SAMPLING = False
RUN_VALIDATION_SAMPLER_PROBE = True
RUN_POST_SAMPLING_CALIBRATION_GRID = False
RUN_TEST_EVAL = True
REGENERATE_FORECASTS = False
VALIDATION_SAMPLER_REGENERATE = False
FAST_DEV_RUN = False

# Optional: set this to an explicit compatible d10a full-checkpoint .pt file.
# By default the notebook uses data/processed/models/diffusion_unet/ddpm_v10a_best.pt.
VALIDATION_CHECKPOINT_PATH_OVERRIDE = None

BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
SAMPLE_BATCH_SIZE = 8
NUM_WORKERS = 0
NUM_EPOCHS = 300
PATIENCE = 40
LEARNING_RATE = 2e-4
MIN_LEARNING_RATE = 2e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
USE_AMP = True

# Conditional U-Net denoiser.
BASE_WIDTH = 48
DROPOUT = 0.05
TIME_EMB_DIM = 128

# Diffusion process and training objective.
DDPM_TIMESTEPS = 1000
COSINE_S = 0.008
USE_AREA_WEIGHTED_LOSS = True
USE_TARGET_WEIGHTED_EPS_LOSS = True
ACTIVE_CELL_LOSS_WEIGHT = float(D10_OBJECTIVE_CONFIG["active_cell_loss_weight"])
TAIL_CELL_LOSS_WEIGHT = float(D10_OBJECTIVE_CONFIG["tail_cell_loss_weight"])
FIRE_SEASON_LOSS_WEIGHT = 1.0
LOSS_WEIGHT_CLIP = 25.0
VAL_LOSS_REPEATS = 4
X0_AUX_LOSS_WEIGHT = float(D10_OBJECTIVE_CONFIG["x0_aux_loss_weight"])
X0_AUX_LOSS_CLIP_MODE = "min_p9999"  # clip reconstructed x0 for auxiliary loss only; options below

# EMA and sampling.
EMA_DECAY = 0.999
USE_EMA_FOR_VALIDATION = True
USE_EMA_FOR_SAMPLING = True
DDIM_STEPS = 100
DDIM_ETA = 0.0
ENSEMBLE_SIZE = 50
VALIDATION_SAMPLER_DDIM_STEPS = 250
VALIDATION_SAMPLER_ENSEMBLE_SIZE = 50

# Final frozen sampler: one model only, labelled d10f.
RUN_TEMPERATURE_SWEEP = False
RUN_DDIM_ETA_PROBE = True
RUN_EMA_WEIGHT_PROBE = False
VALIDATION_XT_SCALES = (1.00,)
VALIDATION_DDIM_ETAS = (0.50,)
VALIDATION_FINAL_Y_CAP = "raw_p999"

# Post-sampling calibration was not selected; keep helpers available but disabled.
POSTCAL_BASE_MODELS = (DIFFUSION_MODEL_NAME,)
POSTCAL_ZERO_BELOW_THRESHOLDS = (0.0, 1e-5, 5e-5, 1e-4, 5e-4)
POSTCAL_POSITIVE_SCALES = (1.0, 1.25, 1.5, 2.0, 3.0)
POSTCAL_FINAL_Y_CAP = "raw_p999"
# DDIM clipping is in transformed target space, not raw burned-fraction space.
# Valid modes: False/"none", True/"manual", "p01_p99", "p001_p999",
# "p001_p9999", "min_p999", "min_p9999", "p01_max", "min_max".
DDIM_CLIP_X0_EACH_STEP = False
TARGET_X0_CLIP_MIN = -8.0
TARGET_X0_CLIP_MAX = 8.0

# Overfit sanity-check controls retained for helper functions. v7 sampler calibration
# does not call train_overfit_debug_model unless RUN_OVERFIT_DEBUG is manually re-enabled.
OVERFIT_N_MONTHS = 6
OVERFIT_NUM_STEPS = 4000
OVERFIT_BATCH_SIZE = 6
OVERFIT_LEARNING_RATE = 5e-4
OVERFIT_EMA_DECAY = 0.999
OVERFIT_LOG_EVERY = 50
OVERFIT_EVAL_EVERY = 250
OVERFIT_ENSEMBLE_SIZE = 10
OVERFIT_DDIM_STEPS = 250
OVERFIT_REGENERATE = True

# Legacy v7 sampler-only calibration. Disabled by default in v9.
V7_SAMPLER_REGENERATE = True
V7_SOURCE_WEIGHT_SOURCES = ("raw",)  # raw behaved better than EMA in v6; add "ema" if needed.
V7_TEMPERATURES = (0.25, 0.50, 0.75, 1.00)
V7_FINAL_X0_CLIP_MODES = (False, "min_p9999", "min_max")
V7_FINAL_Y_CAP_MODES = (False, "raw_p999", "raw_p9999")
V7_PANEL_MAX_VARIANTS = 8
OVERFIT_SAMPLER_VARIANTS = [
    {"label": "raw_noclipx0", "clip_x0": False, "use_ema": False},
    {"label": "raw_x0clip_min_p999", "clip_x0": "min_p999", "use_ema": False},
    {"label": "raw_x0clip_min_p9999", "clip_x0": "min_p9999", "use_ema": False},
    {"label": "raw_x0clip_p001_p9999", "clip_x0": "p001_p9999", "use_ema": False},
    {"label": "raw_x0clip_p01_max", "clip_x0": "p01_max", "use_ema": False},
    {"label": "raw_x0clip_min_max", "clip_x0": "min_max", "use_ema": False},
    {"label": "ema_x0clip_min_p999", "clip_x0": "min_p999", "use_ema": True},
    {"label": "ema_x0clip_min_p9999", "clip_x0": "min_p9999", "use_ema": True},
    {"label": "ema_x0clip_min_max", "clip_x0": "min_max", "use_ema": True},
]
# Backward-compatible alias for older notebook cells/comments.
OVERFIT_CLIP_VARIANTS = OVERFIT_SAMPLER_VARIANTS
ONE_STEP_RECON_TIMESTEPS = (0, 25, 50, 100, 250, 500, 750, 950)
ONE_STEP_RECON_CLIP_MODES = (False, "min_p999", "min_p9999", "p001_p9999", "p01_max", "min_max")
ONE_STEP_RECON_REPEATS = 3
ONE_STEP_RECON_WEIGHT_SOURCES = ("raw", "ema")

# Legacy sampler/debug check from the raw-target notebook. Usually disabled for v4.
RUN_SAMPLER_DEBUG_CHECK = False
SAMPLER_DEBUG_DDIM_STEPS = 250
SAMPLER_DEBUG_ENSEMBLE_SIZE = 10
SAMPLER_DEBUG_TRAIN_N_MONTHS = 6
SAMPLER_DEBUG_REGENERATE = True
SAMPLER_DEBUG_VARIANTS = [
    {"label": "ema_ddim250_noclipx0", "use_ema": True, "clip_x0": False, "steps": SAMPLER_DEBUG_DDIM_STEPS},
    {"label": "raw_ddim250_noclipx0", "use_ema": False, "clip_x0": False, "steps": SAMPLER_DEBUG_DDIM_STEPS},
]

if FAST_DEV_RUN:
    NUM_EPOCHS = 2
    PATIENCE = 2
    DDPM_TIMESTEPS = 100
    DDIM_STEPS = 10
    ENSEMBLE_SIZE = 4
    VALIDATION_SAMPLER_DDIM_STEPS = 10
    VALIDATION_SAMPLER_ENSEMBLE_SIZE = 4
    VAL_LOSS_REPEATS = 1
    OVERFIT_NUM_STEPS = 50
    OVERFIT_EVAL_EVERY = 25
    OVERFIT_ENSEMBLE_SIZE = 4
    OVERFIT_DDIM_STEPS = 10
    print("FAST_DEV_RUN is enabled: using reduced training/sampling settings.")


## 2. Locate processed tensors and output directories

The notebook expects the outputs of the conditioning-tensor notebook under `data/processed/`, unless `WILDFIRE_DATA_DIR` points elsewhere.

In [ ]:
def find_processed_data_dir() -> Path:
    """Find the processed data directory produced by the conditioning-tensor notebook."""
    env = os.environ.get("WILDFIRE_DATA_DIR")
    if env:
        return Path(env).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "data" / "processed",
        cwd.parent / "data" / "processed",
        Path("/mnt/data/data/processed"),
    ]
    for candidate in candidates:
        if (candidate / "splits" / "train.nc").exists():
            return candidate.resolve()
    return (cwd / "data" / "processed").resolve()


DATA_DIR = find_processed_data_dir()
SPLIT_DIR = DATA_DIR / "splits"
SPLIT_FILES = {
    "train": SPLIT_DIR / "train.nc",
    "val": SPLIT_DIR / "val.nc",
    "test": SPLIT_DIR / "test.nc",
}
LSM_PATH = DATA_DIR / "static_lsm.nc"
IBERIAN_MASK_PATH = DATA_DIR / "static_iberian_mask.nc"
NORM_STATS_PATH = DATA_DIR / "norm_stats.json"

MODEL_DIR = DATA_DIR / "models" / "diffusion_unet"
FORECAST_DIR = DATA_DIR / "forecasts" / "diffusion_unet"
METRIC_DIR = DATA_DIR / "metrics"
FIGURE_DIR = DATA_DIR / "figures" / "diffusion_unet"
DETERMINISTIC_FORECAST_DIR = DATA_DIR / "forecasts" / "deterministic_unet"
for path in [MODEL_DIR, FORECAST_DIR, METRIC_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT_PATH = MODEL_DIR / f"{DIFFUSION_EXPERIMENT_ID}_best.pt"
LAST_CHECKPOINT_PATH = MODEL_DIR / f"{DIFFUSION_EXPERIMENT_ID}_last.pt"
OVERFIT_CHECKPOINT_PATH = MODEL_DIR / f"{DIFFUSION_EXPERIMENT_ID}_overfit_debug.pt"
SOURCE_OVERFIT_CHECKPOINT_PATH = MODEL_DIR / f"{SOURCE_OVERFIT_EXPERIMENT_ID}_overfit_debug.pt"
PREVIOUS_FULL_CHECKPOINT_PATH = MODEL_DIR / f"{PREVIOUS_FULL_EXPERIMENT_ID}_best.pt"
SOURCE_FULL_CHECKPOINT_PATH = MODEL_DIR / f"{SOURCE_FULL_EXPERIMENT_ID}_best.pt"
CONFIG_PATH = MODEL_DIR / f"{DIFFUSION_EXPERIMENT_ID}_config.json"
HISTORY_PATH = MODEL_DIR / f"{DIFFUSION_EXPERIMENT_ID}_training_history.csv"
TARGET_TRANSFORM_STATS_PATH = MODEL_DIR / f"{DIFFUSION_EXPERIMENT_ID}_target_transform_stats.json"
if VALIDATION_CHECKPOINT_PATH_OVERRIDE:
    VALIDATION_CHECKPOINT_PATH = Path(VALIDATION_CHECKPOINT_PATH_OVERRIDE).expanduser().resolve()
elif SOURCE_FULL_CHECKPOINT_PATH.exists():
    VALIDATION_CHECKPOINT_PATH = SOURCE_FULL_CHECKPOINT_PATH
elif PREVIOUS_FULL_CHECKPOINT_PATH.exists():
    VALIDATION_CHECKPOINT_PATH = PREVIOUS_FULL_CHECKPOINT_PATH
else:
    VALIDATION_CHECKPOINT_PATH = SOURCE_FULL_CHECKPOINT_PATH
VALIDATION_SAMPLER_MANIFEST_PATH = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_sampler_manifest.csv"

required_paths = [*SPLIT_FILES.values(), LSM_PATH]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing processed tensor files. Expected files under "
        f"{DATA_DIR}. Missing:\n" + "\n".join(missing)
    )

print(f"Using DATA_DIR   = {DATA_DIR}")
print(f"Experiment       = {DIFFUSION_EXPERIMENT_ID} ({D10_OBJECTIVE_CONFIG['description']})")
print(f"Target mode      = {TARGET_MODE}")
print(f"Model dir        = {MODEL_DIR}")
print(f"Forecast dir     = {FORECAST_DIR}")
print(f"Metric dir       = {METRIC_DIR}")
print(f"Figure dir       = {FIGURE_DIR}")
print(f"Best checkpoint  = {BEST_CHECKPOINT_PATH}")
print(f"Validation checkpoint = {VALIDATION_CHECKPOINT_PATH}")
print(f"Previous full checkpoint = {PREVIOUS_FULL_CHECKPOINT_PATH}")
print(f"v6 source overfit checkpoint = {SOURCE_OVERFIT_CHECKPOINT_PATH}")
for name, path in SPLIT_FILES.items():
    print(f"{name:>5}: {path}")
print(f"  lsm: {LSM_PATH}")
print(f" mask: {IBERIAN_MASK_PATH}  # used if present; rebuilt if missing")


## 3. Iberian mask, area weights, and split summary

The model keeps the full rectangular tensor and applies the Iberian/land mask in the loss and diagnostics. This matches the earlier notebooks and avoids geometry-specific cropping inside the U-Net.

In [ ]:
def _sort_split_dataset(ds: xr.Dataset) -> xr.Dataset:
    if "lat" in ds.coords:
        ds = ds.sortby("lat")
    return ds


def load_lsm(lsm_path: Path = LSM_PATH) -> xr.DataArray:
    """Load the static land-sea mask as a 2D DataArray sorted by latitude."""
    ds = xr.open_dataset(lsm_path)
    if "lsm" in ds.data_vars:
        da = ds["lsm"]
    else:
        da = ds[list(ds.data_vars)[0]]
    da = da.squeeze(drop=True)
    if "lat" in da.coords:
        da = da.sortby("lat")
    return da.astype("float32")


def _iberian_southern_boundary(lsm: xr.DataArray) -> xr.DataArray:
    """Heuristic southern boundary used in the EDA notebook to remove North African coastal cells."""
    _, lon2d = xr.broadcast(lsm["lat"], lsm["lon"])
    boundary = xr.full_like(lon2d, 36.0, dtype="float32")
    boundary = xr.where((lon2d >= -5.5) & (lon2d < -2.0), 36.25, boundary)
    boundary = xr.where((lon2d >= -2.0) & (lon2d < 0.5), 36.50, boundary)
    boundary = xr.where((lon2d >= 0.5) & (lon2d < 1.5), 38.00, boundary)
    boundary = xr.where(lon2d >= 1.5, 38.50, boundary)
    return boundary.rename("southern_boundary_lat")


def build_iberian_mask(lsm: xr.DataArray, land_threshold: float = LAND_THRESHOLD) -> xr.Dataset:
    """Build the reusable Iberian-domain mask on the processed 0.25° grid."""
    lsm = lsm.sortby("lat") if "lat" in lsm.coords else lsm
    lat2d, _ = xr.broadcast(lsm["lat"], lsm["lon"])
    southern_boundary = _iberian_southern_boundary(lsm)
    mask = ((lsm > land_threshold) & (lat2d >= southern_boundary)).astype("uint8").rename("iberian_mask")
    mask.attrs.update({
        "description": "Iberian-domain mask: ERA5 LSM land cells north of a heuristic southern boundary.",
        "land_threshold": land_threshold,
        "source_lsm": str(LSM_PATH),
    })
    return xr.Dataset({"iberian_mask": mask, "southern_boundary_lat": southern_boundary})


def ensure_iberian_mask(mask_path: Path = IBERIAN_MASK_PATH, lsm_path: Path = LSM_PATH) -> xr.DataArray:
    """Load the EDA mask if present; otherwise rebuild and save it."""
    if mask_path.exists():
        ds = xr.open_dataset(mask_path)
        if "iberian_mask" in ds.data_vars:
            da = ds["iberian_mask"]
        else:
            da = ds[list(ds.data_vars)[0]]
        return da.squeeze(drop=True).sortby("lat")

    lsm = load_lsm(lsm_path)
    mask_ds = build_iberian_mask(lsm)
    mask_ds.to_netcdf(mask_path)
    return mask_ds["iberian_mask"]


def make_area_weights(lsm: xr.DataArray, land_mask: xr.DataArray) -> xr.DataArray:
    """Approximate cell-area weights: continuous LSM × cos(latitude), normalised."""
    lat_weights = xr.DataArray(
        np.cos(np.deg2rad(lsm["lat"].values)).astype("float64"),
        dims=("lat",),
        coords={"lat": lsm["lat"]},
    )
    weights = (lsm.clip(0, 1).astype("float64") * lat_weights).where(land_mask)
    total = weights.sum(skipna=True)
    if not np.isfinite(float(total)) or float(total) <= 0:
        raise ValueError("Area weights have zero/invalid total. Check LSM and mask alignment.")
    return (weights / total).rename("area_weight")


def describe_split_file(path: Path) -> dict:
    ds = xr.open_dataset(path)
    try:
        times = pd.DatetimeIndex(ds["time"].values)
        return {
            "path": str(path),
            "n_time": int(ds.sizes.get("time", 0)),
            "n_channel": int(ds.sizes.get("channel", 0)),
            "height": int(ds.sizes.get("lat", 0)),
            "width": int(ds.sizes.get("lon", 0)),
            "time_start": str(times.min().date()) if len(times) else None,
            "time_end": str(times.max().date()) if len(times) else None,
            "years": ", ".join(map(str, sorted(set(times.year)))) if len(times) else "",
            "data_vars": ", ".join(ds.data_vars),
        }
    finally:
        ds.close()


lsm = load_lsm(LSM_PATH)
iberian_mask_da = ensure_iberian_mask(IBERIAN_MASK_PATH, LSM_PATH)
iberian_mask_da = iberian_mask_da.reindex(lat=lsm.lat, lon=lsm.lon, method="nearest")
land_mask = (iberian_mask_da > 0).rename("land_mask")
area_weights = make_area_weights(lsm, land_mask)

split_summary = pd.DataFrame([describe_split_file(p) for p in SPLIT_FILES.values()], index=SPLIT_FILES.keys())
display(split_summary)
print(f"Iberian mask cells: {int(land_mask.sum().values):,} / {land_mask.size:,}")
print(f"Area-weight total: {float(area_weights.sum(skipna=True).values):.6f}")

## 4. Diffusion dataset and target scaling

The current diffusion target is `log1p_zscore`: raw burned fraction is transformed as `log1p(y / 1e-2)` and z-scored using train-split target statistics. Outputs are inverse-transformed back to burned fraction and clipped to the physical `[0, 1]` range before scoring.

The epsilon-prediction loss is land/area weighted and uses mild active/tail weights (`2/2`). v10 specifically tunes the reconstruction auxiliary term: default `d10a` uses `x0_aux_loss_weight = 0.10`, while optional `d10b` uses `0.20`. 


In [ ]:
def open_target_split(split_name: str) -> xr.DataArray:
    """Open one split's target as float32 with a split coordinate."""
    ds = _sort_split_dataset(xr.open_dataset(SPLIT_FILES[split_name]))
    if "target" not in ds.data_vars:
        raise KeyError(f"{SPLIT_FILES[split_name]} does not contain a 'target' variable")
    da = ds["target"].astype("float32")
    da = da.assign_coords(split=("time", np.repeat(split_name, da.sizes["time"])))
    return da.rename("target")


def compute_train_tail_threshold(train_target: xr.DataArray, mask: xr.DataArray, q: float = EXTREME_Q) -> float:
    values = train_target.where(mask).values.astype("float64")
    threshold = float(np.nanquantile(values, q))
    if not np.isfinite(threshold):
        raise ValueError("Could not compute train tail threshold.")
    return threshold


def compute_target_transform_stats(
    train_target: xr.DataArray,
    mask: xr.DataArray,
    target_mode: str = TARGET_MODE,
    y_scale: float = LOG1P_TARGET_Y_SCALE,
) -> dict:
    """Train-only target-transform statistics over Iberian land cells."""
    values = np.asarray(train_target.where(mask).values, dtype="float64")
    values = values[np.isfinite(values)]
    values = np.clip(values, 0.0, 1.0)
    if values.size == 0:
        raise ValueError("No finite train target values under the Iberian land mask.")

    if target_mode == "minus1_to_plus1":
        transformed = values * 2.0 - 1.0
        stats = {
            "target_mode": target_mode,
            "mean": 0.0,
            "std": 1.0,
            "y_scale": None,
            "log_mean": None,
            "log_std": None,
            "log_max_for_y_1": None,
        }
    elif target_mode == "log1p_zscore":
        if y_scale <= 0:
            raise ValueError("LOG1P_TARGET_Y_SCALE must be positive.")
        transformed = np.log1p(values / y_scale)
        mean = float(np.mean(transformed))
        std = float(np.std(transformed))
        if not np.isfinite(std) or std <= 1e-12:
            raise ValueError(f"Invalid target transform std: {std}")
        transformed_z = (transformed - mean) / std
        stats = {
            "target_mode": target_mode,
            "y_scale": float(y_scale),
            "log_mean": mean,
            "log_std": std,
            "log_max_for_y_1": float(np.log1p(1.0 / y_scale)),
            "n_train_land_values": int(values.size),
            "raw_y_mean": float(np.mean(values)),
            "raw_y_std": float(np.std(values)),
            "raw_y_p50": float(np.percentile(values, 50)),
            "raw_y_p95": float(np.percentile(values, 95)),
            "raw_y_p99": float(np.percentile(values, 99)),
            "raw_y_p999": float(np.percentile(values, 99.9)),
            "raw_y_p9999": float(np.percentile(values, 99.99)),
            "raw_y_max": float(np.max(values)),
            "transformed_mean": mean,
            "transformed_std": std,
            "transformed_min": float(np.min(transformed_z)),
            "transformed_p0001": float(np.percentile(transformed_z, 0.01)),
            "transformed_p001": float(np.percentile(transformed_z, 0.1)),
            "transformed_p01": float(np.percentile(transformed_z, 1)),
            "transformed_p50": float(np.percentile(transformed_z, 50)),
            "transformed_p99": float(np.percentile(transformed_z, 99)),
            "transformed_p999": float(np.percentile(transformed_z, 99.9)),
            "transformed_p9999": float(np.percentile(transformed_z, 99.99)),
            "transformed_max": float(np.max(transformed_z)),
        }
    else:
        raise ValueError(f"Unsupported TARGET_MODE: {target_mode}")
    return stats


def _stats_scalar(name: str, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    return torch.tensor(float(target_transform_stats[name]), device=device, dtype=dtype)


def target_to_diffusion_scale(y: torch.Tensor) -> torch.Tensor:
    """Map burned fraction y in [0, 1] to diffusion model space."""
    if TARGET_MODE == "minus1_to_plus1":
        return y.clamp(0.0, 1.0).mul(2.0).sub(1.0)
    if TARGET_MODE == "log1p_zscore":
        y_scale = _stats_scalar("y_scale", y.device, y.dtype)
        log_mean = _stats_scalar("log_mean", y.device, y.dtype)
        log_std = _stats_scalar("log_std", y.device, y.dtype)
        z = torch.log1p(y.clamp_min(0.0) / y_scale)
        return (z - log_mean) / log_std
    raise ValueError(f"Unsupported TARGET_MODE: {TARGET_MODE}")


def diffusion_to_target_scale(x: torch.Tensor) -> torch.Tensor:
    """Map diffusion model space back to burned fraction in [0, 1]."""
    if TARGET_MODE == "minus1_to_plus1":
        y = x.add(1.0).mul(0.5)
        return y.clamp(0.0, 1.0) if CLAMP_FINAL_INVERSE_TO_BURN_RANGE else y
    if TARGET_MODE == "log1p_zscore":
        y_scale = _stats_scalar("y_scale", x.device, x.dtype)
        log_mean = _stats_scalar("log_mean", x.device, x.dtype)
        log_std = _stats_scalar("log_std", x.device, x.dtype)
        log_max = _stats_scalar("log_max_for_y_1", x.device, x.dtype)
        log_y = x * log_std + log_mean
        if CLAMP_FINAL_INVERSE_TO_BURN_RANGE:
            log_y = log_y.clamp(0.0, log_max)
        y = y_scale * torch.expm1(log_y)
        return y.clamp(0.0, 1.0) if CLAMP_FINAL_INVERSE_TO_BURN_RANGE else y
    raise ValueError(f"Unsupported TARGET_MODE: {TARGET_MODE}")


def target_x0_clip_bounds(clip_x0: bool | str | None) -> tuple[float, float] | None:
    """Return transformed-space x0 clipping bounds.

    These bounds are deliberately in diffusion target space, not raw burned-fraction
    space. They are used to test whether the DDIM reverse chain is exploding before
    the final inverse transform.
    """
    if clip_x0 is False or clip_x0 is None:
        return None
    if isinstance(clip_x0, str) and clip_x0.lower() in {"", "false", "none", "no", "off"}:
        return None
    if TARGET_MODE == "minus1_to_plus1":
        return (-1.0, 1.0)

    mode = "manual" if clip_x0 is True else str(clip_x0).lower()
    if mode == "manual":
        return (float(TARGET_X0_CLIP_MIN), float(TARGET_X0_CLIP_MAX))
    if mode == "p01_p99":
        return (float(target_transform_stats["transformed_p01"]), float(target_transform_stats["transformed_p99"]))
    if mode == "p001_p999":
        return (float(target_transform_stats["transformed_p001"]), float(target_transform_stats["transformed_p999"]))
    if mode == "p001_p9999":
        return (float(target_transform_stats["transformed_p001"]), float(target_transform_stats["transformed_p9999"]))
    if mode == "p0001_p9999":
        return (float(target_transform_stats["transformed_p0001"]), float(target_transform_stats["transformed_p9999"]))
    if mode == "min_p999":
        return (float(target_transform_stats["transformed_min"]), float(target_transform_stats["transformed_p999"]))
    if mode == "min_p9999":
        return (float(target_transform_stats["transformed_min"]), float(target_transform_stats["transformed_p9999"]))
    if mode == "p01_max":
        return (float(target_transform_stats["transformed_p01"]), float(target_transform_stats["transformed_max"]))
    if mode == "min_max":
        return (float(target_transform_stats["transformed_min"]), float(target_transform_stats["transformed_max"]))
    raise ValueError(f"Unknown x0 clipping mode: {clip_x0!r}")


def clip_predicted_x0_if_requested(pred_x0: torch.Tensor, clip_x0: bool | str | None) -> torch.Tensor:
    """Optional DDIM x0 clipping in diffusion space, not burned-fraction space."""
    bounds = target_x0_clip_bounds(clip_x0)
    if bounds is None:
        return pred_x0
    lo, hi = bounds
    return pred_x0.clamp(lo, hi)


def make_loss_weight_array(
    target: xr.DataArray,
    weights_2d: xr.DataArray,
    train_tail_threshold: float,
    use_area_weighted_loss: bool = USE_AREA_WEIGHTED_LOSS,
    use_target_weighted_eps_loss: bool = USE_TARGET_WEIGHTED_EPS_LOSS,
) -> np.ndarray:
    """Build per-sample loss weights with land/area mask and optional mild target weighting."""
    target_np = np.asarray(target.transpose("time", "lat", "lon").values, dtype="float32")
    land_np = np.asarray(land_mask.transpose("lat", "lon").values, dtype=bool)
    if use_area_weighted_loss:
        base_2d = np.asarray(weights_2d.transpose("lat", "lon").fillna(0).values, dtype="float32")
    else:
        base_2d = land_np.astype("float32")
        base_2d = base_2d / max(base_2d.sum(), 1.0)
    base = np.broadcast_to(base_2d[None, :, :], target_np.shape).copy()
    base[:, ~land_np] = 0.0

    if use_target_weighted_eps_loss:
        multiplier = np.ones_like(base, dtype="float32")
        active = target_np > BURN_THRESHOLD
        tail = target_np >= train_tail_threshold
        multiplier = np.where(active, ACTIVE_CELL_LOSS_WEIGHT, multiplier)
        multiplier = np.where(tail, TAIL_CELL_LOSS_WEIGHT, multiplier)
        if FIRE_SEASON_LOSS_WEIGHT != 1.0:
            months = pd.DatetimeIndex(target.time.values).month
            fire_season = np.isin(months, FIRE_SEASON_MONTHS)[:, None, None]
            multiplier = np.where(fire_season, multiplier * FIRE_SEASON_LOSS_WEIGHT, multiplier)
        multiplier = np.clip(multiplier, 0.0, LOSS_WEIGHT_CLIP)
        base *= multiplier

    return base.astype("float32")


def _channel_names_from_dataset(ds: xr.Dataset) -> list[str]:
    if "channel" in ds.coords:
        return [str(v) for v in ds["channel"].values]
    n_channel = int(ds["conditioning"].sizes["channel"])
    return [f"channel_{i}" for i in range(n_channel)]


def nonfinite_count(array: np.ndarray) -> int:
    """Count NaN/inf values in a numpy array."""
    return int((~np.isfinite(array)).sum())


def fill_tensor_nans(array: np.ndarray, fill_value: float = 0.0) -> np.ndarray:
    """Replace NaN/inf values before tensors enter the U-Net.

    The processed tensors can legitimately contain NaNs over ocean/excluded cells.
    The loss and metrics are masked to Iberian land, but convolutions cannot carry NaNs.
    For z-scored channels, 0.0 is the train-mean value; outside the positive-loss mask it is harmless.
    """
    return np.nan_to_num(array, nan=fill_value, posinf=fill_value, neginf=fill_value).astype("float32")


class WildfireDiffusionDataset(Dataset):
    """Small in-memory dataset for monthly conditioning tensors and burned-fraction targets."""

    def __init__(self, split_name: str, train_tail_threshold: float):
        self.split_name = split_name
        self.path = SPLIT_FILES[split_name]
        ds = _sort_split_dataset(xr.open_dataset(self.path)).load()
        if "conditioning" not in ds.data_vars or "target" not in ds.data_vars:
            raise KeyError(f"{self.path} must contain 'conditioning' and 'target'.")

        self.ds = ds
        self.channels = _channel_names_from_dataset(ds)
        self.times = pd.DatetimeIndex(ds["time"].values)
        self.lat = ds["lat"].values
        self.lon = ds["lon"].values

        conditioning_raw = np.asarray(
            ds["conditioning"].transpose("time", "channel", "lat", "lon").values,
            dtype="float32",
        )
        target_da = ds["target"].transpose("time", "lat", "lon").astype("float32")
        target_raw = np.asarray(target_da.values, dtype="float32")
        self.loss_weight = make_loss_weight_array(target_da, area_weights, train_tail_threshold)

        # Align the land mask to this split before diagnostics.
        split_land = land_mask.reindex(lat=target_da.lat, lon=target_da.lon, method="nearest").astype(bool)
        land_np = np.asarray(split_land.transpose("lat", "lon").values, dtype=bool)
        land_cond_np = np.broadcast_to(land_np[None, None, :, :], conditioning_raw.shape)
        land_target_np = np.broadcast_to(land_np[None, :, :], target_raw.shape)
        positive_loss_np = self.loss_weight > 0
        positive_loss_cond_np = np.broadcast_to(positive_loss_np[:, None, :, :], conditioning_raw.shape)

        self.quality = {
            "split": split_name,
            "conditioning_nonfinite_total": nonfinite_count(conditioning_raw),
            "conditioning_nonfinite_on_land": int((~np.isfinite(conditioning_raw) & land_cond_np).sum()),
            "conditioning_nonfinite_with_positive_loss_weight": int((~np.isfinite(conditioning_raw) & positive_loss_cond_np).sum()),
            "target_nonfinite_total": nonfinite_count(target_raw),
            "target_nonfinite_on_land": int((~np.isfinite(target_raw) & land_target_np).sum()),
            "target_nonfinite_with_positive_loss_weight": int((~np.isfinite(target_raw) & positive_loss_np).sum()),
        }

        # Fill after diagnostics, before PyTorch sees the tensors.
        self.conditioning = fill_tensor_nans(conditioning_raw, fill_value=0.0)
        self.target = fill_tensor_nans(target_raw, fill_value=0.0)

        # Targets must still be valid where the loss is positive; otherwise the labels/mask are bad.
        if self.quality["target_nonfinite_with_positive_loss_weight"] > 0:
            raise ValueError(
                f"Target has non-finite values on positive-weight Iberian cells in {self.path}. "
                "Check target aggregation/mask alignment before training."
            )
        if not np.isfinite(self.conditioning).all() or not np.isfinite(self.target).all():
            raise ValueError(f"NaN/inf fill failed for {self.path}")

    def __len__(self) -> int:
        return int(self.conditioning.shape[0])

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor | str]:
        cond = torch.from_numpy(self.conditioning[idx])
        y = torch.from_numpy(self.target[idx][None, :, :])
        x0 = target_to_diffusion_scale(y)
        weight = torch.from_numpy(self.loss_weight[idx][None, :, :])
        return {
            "conditioning": cond,
            "x0": x0,
            "target": y,
            "loss_weight": weight,
            "time": str(self.times[idx].date()),
        }


targets = {split: open_target_split(split) for split in ["train", "val", "test"]}
target_all = xr.concat([targets[s] for s in ["train", "val", "test"]], dim="time").sortby("time")
train_tail_threshold = compute_train_tail_threshold(targets["train"], land_mask, EXTREME_Q)
target_transform_stats = compute_target_transform_stats(targets["train"], land_mask, TARGET_MODE, LOG1P_TARGET_Y_SCALE)
TARGET_TRANSFORM_STATS_PATH.write_text(json.dumps(target_transform_stats, indent=2, default=str))
print(f"Train top-1% burned-fraction threshold: {train_tail_threshold:.8g}")
print(f"Saved target transform stats: {TARGET_TRANSFORM_STATS_PATH}")
display(pd.DataFrame([target_transform_stats]))

train_dataset = WildfireDiffusionDataset("train", train_tail_threshold)
val_dataset = WildfireDiffusionDataset("val", train_tail_threshold)
test_dataset = WildfireDiffusionDataset("test", train_tail_threshold)

quality_df = pd.DataFrame([train_dataset.quality, val_dataset.quality, test_dataset.quality])
print("Non-finite tensor diagnostics before NaN/inf fill:")
display(quality_df)
if (quality_df["conditioning_nonfinite_with_positive_loss_weight"] > 0).any():
    print(
        "WARNING: some conditioning values were non-finite on positive-loss Iberian cells and were filled with 0. "
        "This is acceptable for a first run, but inspect affected channels if validation behaviour looks odd."
    )

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
train_eval_loader = DataLoader(train_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
test_loader = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

print(f"Dataset sizes: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")
print(f"Conditioning channels: {len(train_dataset.channels)}")
print(train_dataset.channels)


## 5. Conditional U-Net denoiser

The noisy burned-fraction map is the model state. The processed climate/static/calendar tensor is concatenated to the feature map at every resolution. At lower resolutions the conditioning tensor is bilinearly downsampled to the current feature size before concatenation.

This keeps the conditioning pathway explicit and easy to audit: no hidden cross-attention or implicit conditioning projection is needed for this small 32 × 54 grid.

In [ ]:
def group_count(num_channels: int, max_groups: int = 8) -> int:
    """Largest GroupNorm group count up to max_groups that divides num_channels."""
    for groups in range(min(max_groups, num_channels), 0, -1):
        if num_channels % groups == 0:
            return groups
    return 1


class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        if half <= 1:
            raise ValueError("Time embedding dimension must be at least 4.")
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(half, device=timesteps.device, dtype=torch.float32) / (half - 1)
        )
        args = timesteps.float()[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb


class ResBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, time_dim: int, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(group_count(in_channels), in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_channels)
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x: torch.Tensor, time_emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_proj(time_emb)[:, :, None, None]
        h = self.conv2(self.dropout(F.silu(self.norm2(h))))
        return h + self.skip(x)


class Downsample(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, stride=2, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.interpolate(x, scale_factor=2.0, mode="bilinear", align_corners=False)
        return self.conv(x)


class ConditionalUNetDenoiser(nn.Module):
    """Two-level conditional U-Net that predicts DDPM epsilon noise."""

    def __init__(
        self,
        cond_channels: int,
        base_width: int = BASE_WIDTH,
        time_dim: int = TIME_EMB_DIM,
        dropout: float = DROPOUT,
        out_channels: int = 1,
    ):
        super().__init__()
        self.cond_channels = int(cond_channels)
        self.base_width = int(base_width)
        self.time_dim = int(time_dim)
        self.dropout = float(dropout)

        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim),
        )

        c = self.cond_channels
        b = base_width
        self.enc1a = ResBlock(1 + c, b, time_dim, dropout)
        self.enc1b = ResBlock(b + c, b, time_dim, dropout)
        self.down1 = Downsample(b)

        self.enc2a = ResBlock(b + c, b * 2, time_dim, dropout)
        self.enc2b = ResBlock(b * 2 + c, b * 2, time_dim, dropout)
        self.down2 = Downsample(b * 2)

        self.mid1 = ResBlock(b * 2 + c, b * 4, time_dim, dropout)
        self.mid2 = ResBlock(b * 4 + c, b * 4, time_dim, dropout)

        self.up2 = Upsample(b * 4, b * 2)
        self.dec2a = ResBlock(b * 2 + b * 2 + c, b * 2, time_dim, dropout)
        self.dec2b = ResBlock(b * 2 + c, b * 2, time_dim, dropout)

        self.up1 = Upsample(b * 2, b)
        self.dec1a = ResBlock(b + b + c, b, time_dim, dropout)
        self.dec1b = ResBlock(b + c, b, time_dim, dropout)

        self.out_norm = nn.GroupNorm(group_count(b), b)
        self.out_conv = nn.Conv2d(b, out_channels, kernel_size=3, padding=1)

    @staticmethod
    def _resize_cond(cond: torch.Tensor, like: torch.Tensor) -> torch.Tensor:
        if cond.shape[-2:] == like.shape[-2:]:
            return cond
        return F.interpolate(cond, size=like.shape[-2:], mode="bilinear", align_corners=False)

    def forward(self, x_t: torch.Tensor, conditioning: torch.Tensor, timesteps: torch.Tensor) -> torch.Tensor:
        time_emb = self.time_mlp(timesteps)

        c0 = self._resize_cond(conditioning, x_t)
        h1 = self.enc1a(torch.cat([x_t, c0], dim=1), time_emb)
        h1 = self.enc1b(torch.cat([h1, c0], dim=1), time_emb)

        h2_in = self.down1(h1)
        c1 = self._resize_cond(conditioning, h2_in)
        h2 = self.enc2a(torch.cat([h2_in, c1], dim=1), time_emb)
        h2 = self.enc2b(torch.cat([h2, c1], dim=1), time_emb)

        h_mid_in = self.down2(h2)
        c2 = self._resize_cond(conditioning, h_mid_in)
        h_mid = self.mid1(torch.cat([h_mid_in, c2], dim=1), time_emb)
        h_mid = self.mid2(torch.cat([h_mid, c2], dim=1), time_emb)

        u2 = self.up2(h_mid)
        if u2.shape[-2:] != h2.shape[-2:]:
            u2 = F.interpolate(u2, size=h2.shape[-2:], mode="bilinear", align_corners=False)
        u2 = self.dec2a(torch.cat([u2, h2, c1], dim=1), time_emb)
        u2 = self.dec2b(torch.cat([u2, c1], dim=1), time_emb)

        u1 = self.up1(u2)
        if u1.shape[-2:] != h1.shape[-2:]:
            u1 = F.interpolate(u1, size=h1.shape[-2:], mode="bilinear", align_corners=False)
        u1 = self.dec1a(torch.cat([u1, h1, c0], dim=1), time_emb)
        u1 = self.dec1b(torch.cat([u1, c0], dim=1), time_emb)

        return self.out_conv(F.silu(self.out_norm(u1)))


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


cond_channels = len(train_dataset.channels)
model_preview = ConditionalUNetDenoiser(cond_channels=cond_channels).to(DEVICE)
with torch.no_grad():
    batch = next(iter(train_loader))
    x0 = batch["x0"].to(DEVICE)
    cond = batch["conditioning"].to(DEVICE)
    t = torch.randint(0, DDPM_TIMESTEPS, (x0.shape[0],), device=DEVICE)
    pred = model_preview(x0, cond, t)
print(model_preview.__class__.__name__)
print(f"Trainable parameters: {count_parameters(model_preview):,}")
print(f"Input x0 shape:       {tuple(x0.shape)}")
print(f"Conditioning shape:   {tuple(cond.shape)}")
print(f"Output epsilon shape: {tuple(pred.shape)}")
del model_preview, batch, x0, cond, t, pred
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 6. Cosine DDPM schedule and epsilon-prediction loss

The forward noising process uses a cosine cumulative-alpha schedule. The model is trained to predict the Gaussian noise `epsilon` added to the scaled burned-fraction target.

In [ ]:
def cosine_beta_schedule(timesteps: int, s: float = COSINE_S) -> torch.Tensor:
    """Cosine schedule from Nichol & Dhariwal-style improved DDPMs."""
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps, dtype=torch.float64)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1.0 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return betas.clamp(1e-5, 0.999).float()


def extract(coefficients: torch.Tensor, timesteps: torch.Tensor, x_shape: tuple[int, ...]) -> torch.Tensor:
    """Gather 1D schedule coefficients and reshape for broadcasting over an image batch."""
    out = coefficients.gather(0, timesteps)
    return out.reshape(timesteps.shape[0], *((1,) * (len(x_shape) - 1)))


class DiffusionSchedule:
    def __init__(self, timesteps: int = DDPM_TIMESTEPS, device: torch.device = DEVICE):
        self.num_timesteps = int(timesteps)
        betas = cosine_beta_schedule(timesteps).to(device)
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        self.betas = betas
        self.alphas = alphas
        self.alphas_cumprod = alphas_cumprod
        self.sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
        self.sqrt_recip_alphas_cumprod = torch.sqrt(1.0 / alphas_cumprod)
        self.sqrt_recipm1_alphas_cumprod = torch.sqrt(1.0 / alphas_cumprod - 1.0)

    def to(self, device: torch.device) -> "DiffusionSchedule":
        for name, value in vars(self).items():
            if torch.is_tensor(value):
                setattr(self, name, value.to(device))
        return self

    def q_sample(self, x0: torch.Tensor, timesteps: torch.Tensor, noise: torch.Tensor | None = None) -> torch.Tensor:
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_alpha_bar = extract(self.sqrt_alphas_cumprod, timesteps, x0.shape)
        sqrt_one_minus_alpha_bar = extract(self.sqrt_one_minus_alphas_cumprod, timesteps, x0.shape)
        return sqrt_alpha_bar * x0 + sqrt_one_minus_alpha_bar * noise

    def predict_x0_from_eps(self, x_t: torch.Tensor, timesteps: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
        return (
            extract(self.sqrt_recip_alphas_cumprod, timesteps, x_t.shape) * x_t
            - extract(self.sqrt_recipm1_alphas_cumprod, timesteps, x_t.shape) * eps
        )


def weighted_mse(pred: torch.Tensor, target: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
    loss_map = (pred - target) ** 2
    weight = weight.to(loss_map.dtype).expand_as(loss_map)
    valid = torch.isfinite(loss_map) & torch.isfinite(weight) & (weight > 0)
    weighted = torch.where(valid, loss_map * weight, torch.zeros_like(loss_map))
    denom = torch.where(valid, weight, torch.zeros_like(weight)).sum().clamp_min(1e-12)
    return weighted.sum() / denom


def weighted_epsilon_mse(pred_eps: torch.Tensor, true_eps: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
    return weighted_mse(pred_eps, true_eps, weight)


def diffusion_epsilon_loss(
    model: nn.Module,
    schedule: DiffusionSchedule,
    batch: dict[str, torch.Tensor],
) -> torch.Tensor:
    conditioning = batch["conditioning"].to(DEVICE, non_blocking=True)
    x0 = batch["x0"].to(DEVICE, non_blocking=True)
    weight = batch["loss_weight"].to(DEVICE, non_blocking=True)
    bsz = x0.shape[0]
    timesteps = torch.randint(0, schedule.num_timesteps, (bsz,), device=DEVICE, dtype=torch.long)
    noise = torch.randn_like(x0)
    x_t = schedule.q_sample(x0, timesteps, noise)
    pred_noise = model(x_t, conditioning, timesteps)
    eps_loss = weighted_epsilon_mse(pred_noise, noise, weight)
    if X0_AUX_LOSS_WEIGHT and X0_AUX_LOSS_WEIGHT > 0:
        pred_x0 = schedule.predict_x0_from_eps(x_t, timesteps, pred_noise)
        pred_x0_for_loss = clip_predicted_x0_if_requested(pred_x0, X0_AUX_LOSS_CLIP_MODE)
        x0_loss = weighted_mse(pred_x0_for_loss, x0, weight)
        return eps_loss + float(X0_AUX_LOSS_WEIGHT) * x0_loss
    return eps_loss


schedule = DiffusionSchedule(DDPM_TIMESTEPS, DEVICE)
print(f"DDPM timesteps: {schedule.num_timesteps}")
print(f"beta range:     {float(schedule.betas.min()):.3g} .. {float(schedule.betas.max()):.3g}")
print(f"alpha_bar_T:    {float(schedule.alphas_cumprod[-1]):.3g}")


## 7. Optimizer, EMA, checkpointing, and training loop

Validation loss is computed using EMA weights by default. The checkpoint stores both raw model weights and EMA weights; sampling later uses EMA weights unless `USE_EMA_FOR_SAMPLING = False`.

In [ ]:
class EMA:
    """Exponential moving average of trainable model parameters."""

    def __init__(self, model: nn.Module, decay: float = EMA_DECAY):
        self.decay = float(decay)
        self.shadow = {
            name: param.detach().clone()
            for name, param in model.named_parameters()
            if param.requires_grad
        }
        self.backup: dict[str, torch.Tensor] = {}

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            if name not in self.shadow:
                self.shadow[name] = param.detach().clone()
            else:
                self.shadow[name].mul_(self.decay).add_(param.detach(), alpha=1.0 - self.decay)

    def state_dict(self) -> dict[str, torch.Tensor]:
        return {name: value.detach().cpu().clone() for name, value in self.shadow.items()}

    def load_state_dict(self, state_dict: dict[str, torch.Tensor], device: torch.device = DEVICE) -> None:
        self.shadow = {name: value.detach().to(device).clone() for name, value in state_dict.items()}

    @torch.no_grad()
    def store(self, model: nn.Module) -> None:
        self.backup = {
            name: param.detach().clone()
            for name, param in model.named_parameters()
            if param.requires_grad
        }

    @torch.no_grad()
    def copy_to(self, model: nn.Module) -> None:
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                param.data.copy_(self.shadow[name].data)

    @torch.no_grad()
    def restore(self, model: nn.Module) -> None:
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data.copy_(self.backup[name].data)
        self.backup = {}


def autocast_context():
    if DEVICE.type == "cuda" and USE_AMP:
        return torch.cuda.amp.autocast()
    return nullcontext()


def make_model_config() -> dict:
    return {
        "model_name": "conditional_ddpm_unet",
        "cond_channels": cond_channels,
        "channel_names": train_dataset.channels,
        "base_width": BASE_WIDTH,
        "dropout": DROPOUT,
        "time_emb_dim": TIME_EMB_DIM,
        "ddpm_timesteps": DDPM_TIMESTEPS,
        "cosine_s": COSINE_S,
        "target_mode": TARGET_MODE,
        "target_transform_stats": target_transform_stats,
        "log1p_target_y_scale": LOG1P_TARGET_Y_SCALE,
        "clamp_final_inverse_to_burn_range": CLAMP_FINAL_INVERSE_TO_BURN_RANGE,
        "use_area_weighted_loss": USE_AREA_WEIGHTED_LOSS,
        "use_target_weighted_eps_loss": USE_TARGET_WEIGHTED_EPS_LOSS,
        "active_cell_loss_weight": ACTIVE_CELL_LOSS_WEIGHT,
        "tail_cell_loss_weight": TAIL_CELL_LOSS_WEIGHT,
        "fire_season_loss_weight": FIRE_SEASON_LOSS_WEIGHT,
        "loss_weight_clip": LOSS_WEIGHT_CLIP,
        "x0_aux_loss_weight": X0_AUX_LOSS_WEIGHT,
        "x0_aux_loss_clip_mode": X0_AUX_LOSS_CLIP_MODE,
        "target_x0_clip_min": TARGET_X0_CLIP_MIN,
        "target_x0_clip_max": TARGET_X0_CLIP_MAX,
        "ddim_clip_x0_each_step": DDIM_CLIP_X0_EACH_STEP,
        "train_tail_threshold": train_tail_threshold,
        "learning_rate": LEARNING_RATE,
        "min_learning_rate": MIN_LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "ema_decay": EMA_DECAY,
        "ddim_steps_default": DDIM_STEPS,
        "ddim_eta_default": DDIM_ETA,
        "ensemble_size_default": ENSEMBLE_SIZE,
        "random_seed": RANDOM_SEED,
    }


def save_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, default=str))


def save_checkpoint(
    path: Path,
    model: nn.Module,
    ema: EMA,
    optimizer: torch.optim.Optimizer,
    scheduler_lr: torch.optim.lr_scheduler._LRScheduler,
    epoch: int,
    best_val_loss: float,
    history: list[dict],
    config: dict,
) -> None:
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "ema_state_dict": ema.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "lr_scheduler_state_dict": scheduler_lr.state_dict(),
            "best_val_loss": best_val_loss,
            "history": history,
            "config": config,
        },
        path,
    )


def load_checkpoint_if_available(
    path: Path,
    model: nn.Module,
    ema: EMA,
    optimizer: torch.optim.Optimizer,
    scheduler_lr: torch.optim.lr_scheduler._LRScheduler,
) -> tuple[int, float, list[dict]]:
    if not path.exists():
        return 1, math.inf, []
    ckpt = torch.load(path, map_location=DEVICE)
    ckpt_cfg = ckpt.get("config", {})
    if ckpt_cfg.get("target_mode", TARGET_MODE) != TARGET_MODE:
        raise ValueError(
            f"Checkpoint target_mode={ckpt_cfg.get('target_mode')} does not match current TARGET_MODE={TARGET_MODE}. "
            "Use the experiment-specific v4 checkpoint path or disable RESUME_TRAINING."
        )
    model.load_state_dict(ckpt["model_state_dict"])
    if "ema_state_dict" in ckpt:
        ema.load_state_dict(ckpt["ema_state_dict"], DEVICE)
    if "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if "lr_scheduler_state_dict" in ckpt:
        scheduler_lr.load_state_dict(ckpt["lr_scheduler_state_dict"])
    start_epoch = int(ckpt.get("epoch", 0)) + 1
    best_val_loss = float(ckpt.get("best_val_loss", math.inf))
    history = list(ckpt.get("history", []))
    print(f"Resumed from {path}: next epoch={start_epoch}, best_val_loss={best_val_loss:.8f}")
    return start_epoch, best_val_loss, history


def run_train_epoch(
    model: nn.Module,
    schedule: DiffusionSchedule,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
    ema: EMA,
    max_batches: int | None = None,
) -> float:
    model.train()
    running_loss = 0.0
    n_batches = 0
    pbar = tqdm(loader, leave=False, desc="train")
    for batch_idx, batch in enumerate(pbar, start=1):
        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            loss = diffusion_epsilon_loss(model, schedule, batch)
        scaler.scale(loss).backward()
        if GRAD_CLIP_NORM is not None:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)

        running_loss += float(loss.detach().cpu())
        n_batches += 1
        pbar.set_postfix(loss=running_loss / n_batches)
        if max_batches is not None and batch_idx >= max_batches:
            break
    return running_loss / max(n_batches, 1)


@torch.no_grad()
def run_validation_loss(
    model: nn.Module,
    schedule: DiffusionSchedule,
    loader: DataLoader,
    repeats: int = VAL_LOSS_REPEATS,
    max_batches: int | None = None,
) -> float:
    model.eval()
    losses = []
    for repeat in range(repeats):
        for batch_idx, batch in enumerate(tqdm(loader, leave=False, desc=f"val eps loss {repeat+1}/{repeats}"), start=1):
            loss = diffusion_epsilon_loss(model, schedule, batch)
            losses.append(float(loss.detach().cpu()))
            if max_batches is not None and batch_idx >= max_batches:
                break
    return float(np.mean(losses)) if losses else math.inf


In [ ]:
config = make_model_config()
save_json(CONFIG_PATH, config)

model = ConditionalUNetDenoiser(
    cond_channels=cond_channels,
    base_width=BASE_WIDTH,
    time_dim=TIME_EMB_DIM,
    dropout=DROPOUT,
).to(DEVICE)
ema = EMA(model, EMA_DECAY)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=MIN_LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda" and USE_AMP))
schedule = DiffusionSchedule(DDPM_TIMESTEPS, DEVICE)

start_epoch, best_val_loss, history_records = (1, math.inf, [])
if RESUME_TRAINING:
    start_epoch, best_val_loss, history_records = load_checkpoint_if_available(
        LAST_CHECKPOINT_PATH, model, ema, optimizer, lr_scheduler
    )

print(f"Model parameters: {count_parameters(model):,}")
print(f"Config saved:     {CONFIG_PATH}")

In [ ]:
if RUN_TRAIN and not (RUN_OVERFIT_DEBUG and STOP_AFTER_OVERFIT_DEBUG):
    no_improvement = 0
    max_train_batches = 2 if FAST_DEV_RUN else None
    max_val_batches = 2 if FAST_DEV_RUN else None

    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        t0 = time.time()
        train_loss = run_train_epoch(
            model, schedule, train_loader, optimizer, scaler, ema, max_batches=max_train_batches
        )

        if USE_EMA_FOR_VALIDATION:
            ema.store(model)
            ema.copy_to(model)
        val_loss = run_validation_loss(schedule=schedule, model=model, loader=val_loader, max_batches=max_val_batches)
        if USE_EMA_FOR_VALIDATION:
            ema.restore(model)

        lr_scheduler.step()
        elapsed = time.time() - t0
        row = {
            "epoch": epoch,
            "train_eps_mse": train_loss,
            "val_eps_mse": val_loss,
            "best_val_eps_mse": min(best_val_loss, val_loss),
            "lr": optimizer.param_groups[0]["lr"],
            "elapsed_seconds": elapsed,
            "used_ema_for_validation": USE_EMA_FOR_VALIDATION,
        }
        history_records.append(row)
        pd.DataFrame(history_records).to_csv(HISTORY_PATH, index=False)

        improved = val_loss < best_val_loss
        if improved:
            best_val_loss = val_loss
            no_improvement = 0
            save_checkpoint(BEST_CHECKPOINT_PATH, model, ema, optimizer, lr_scheduler, epoch, best_val_loss, history_records, config)
        else:
            no_improvement += 1

        save_checkpoint(LAST_CHECKPOINT_PATH, model, ema, optimizer, lr_scheduler, epoch, best_val_loss, history_records, config)

        print(
            f"epoch {epoch:03d} | train={train_loss:.8f} | val={val_loss:.8f} | "
            f"best={best_val_loss:.8f} | lr={optimizer.param_groups[0]['lr']:.2e} | "
            f"{'saved best' if improved else f'patience {no_improvement}/{PATIENCE}'} | {elapsed:.1f}s"
        )

        if no_improvement >= PATIENCE:
            print(f"Early stopping after {epoch} epochs.")
            break
else:
    if RUN_OVERFIT_DEBUG and STOP_AFTER_OVERFIT_DEBUG:
        print("STOP_AFTER_OVERFIT_DEBUG=True: skipping full training until the overfit sanity check is inspected.")
    else:
        print("RUN_TRAIN=False: skipping training. Existing checkpoint will be used for sampling/evaluation if available.")

if HISTORY_PATH.exists():
    history_df = pd.read_csv(HISTORY_PATH)
    display(history_df.tail(10))
else:
    history_df = pd.DataFrame(history_records)

## 8. DDIM sampler and ensemble generation

The v10 validation probe is compact and centred on the best v9 practical sampler. The primary candidate is `t100_e05`: raw weights, `x_T_scale=1.00`, `ddim_eta=0.50`, no per-step x0 clipping, and final raw-space `p999` cap. Secondary variants check `eta=0`, slightly lower `x_T_scale`, and EMA collapse.


In [ ]:


def load_forecast_from_netcdf(path: Path) -> xr.DataArray:
    """Load a forecast NetCDF fully into memory and close the file handle.

    This avoids Windows PermissionError issues when the same notebook later
    regenerates a forecast with the same path.
    """
    with xr.open_dataset(path) as ds:
        da = ds["forecast"] if "forecast" in ds.data_vars else ds[list(ds.data_vars)[0]]
        da = da.load()
    return da.astype("float32")


WINDOWS_SAFE_NETCDF_PATH_LIMIT = 240


def _path_length(path: Path) -> int:
    """Approximate the path length seen by Windows APIs/netCDF4."""
    return len(str(Path(path)))


def _path_is_too_long_for_netcdf(path: Path, limit: int = WINDOWS_SAFE_NETCDF_PATH_LIMIT) -> bool:
    """Be conservative because netCDF4 on Windows can still hit MAX_PATH."""
    return _path_length(path) >= int(limit)


def _compact_path_stem(stem: str, max_chars: int = 96) -> str:
    """Return a filesystem-safe, short stem for fallback/rerun files."""
    safe = "".join(ch if ch.isalnum() or ch in {"-", "_"} else "_" for ch in str(stem))
    if max_chars <= 0:
        return "forecast"
    if len(safe) <= max_chars:
        return safe
    digest = uuid.uuid5(uuid.NAMESPACE_URL, safe).hex[:8]
    keep = max(8, max_chars - len(digest) - 1)
    return f"{safe[:keep]}_{digest}"


def _short_temp_netcdf_path(out_path: Path) -> Path:
    """Create a short same-directory temporary path for NetCDF writes."""
    out_path = Path(out_path)
    for _ in range(100):
        candidate = out_path.parent / f"_tmp_{uuid.uuid4().hex[:12]}{out_path.suffix or '.nc'}"
        if not candidate.exists():
            return candidate
    raise RuntimeError(f"Could not allocate a temporary NetCDF path under {out_path.parent}")


def _short_netcdf_path(out_path: Path, reason: str = "shortpath", unique: bool = False) -> Path:
    """Return a short NetCDF path next to the requested output.

    The v7 calibration filenames can exceed Windows' practical 260-character
    limit once the project directory is included. This function keeps the path
    comfortably under WINDOWS_SAFE_NETCDF_PATH_LIMIT while preserving a hash of
    the originally requested path. With unique=False it is deterministic, so
    cache loading still works across notebook reruns.
    """
    out_path = Path(out_path)
    suffix = out_path.suffix or ".nc"
    digest = uuid.uuid5(uuid.NAMESPACE_URL, str(out_path)).hex[:10]
    unique_part = f"_{time.strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}" if unique else ""
    trailer = f"_{reason}_{digest}{unique_part}{suffix}"
    parent_len = _path_length(out_path.parent)
    # +1 for the path separator between parent and filename.
    max_stem = WINDOWS_SAFE_NETCDF_PATH_LIMIT - parent_len - 1 - len(trailer)
    if max_stem < 8:
        filename = f"fc_{reason}_{digest}{unique_part}{suffix}"
    else:
        stem = _compact_path_stem(out_path.stem, max_chars=min(48, max_stem))
        filename = f"{stem}{trailer}"
    return out_path.with_name(filename)


def resolve_netcdf_cache_path(out_path: Path) -> Path:
    """Return the path that should be used for cache lookup/writing."""
    out_path = Path(out_path)
    if _path_is_too_long_for_netcdf(out_path):
        return _short_netcdf_path(out_path, reason="shortpath", unique=False)
    return out_path


def _fallback_netcdf_path(out_path: Path, reason: str = "rerun") -> Path:
    """Return a unique, short fallback path next to the requested output."""
    return _short_netcdf_path(out_path, reason=reason, unique=True)


def _clear_xarray_file_cache() -> None:
    """Best-effort close of xarray/netCDF file managers before overwriting files."""
    try:
        from xarray.backends.file_manager import FILE_CACHE
        FILE_CACHE.clear()
    except Exception:
        pass


def _write_dataset_to_netcdf(ds: xr.Dataset, path: Path) -> None:
    """Write using netCDF4 when available, with a portability fallback."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        ds.to_netcdf(path, mode="w", engine="netcdf4")
    except (ImportError, ModuleNotFoundError) as exc:
        # The project environment uses netCDF4, but this keeps the notebook
        # usable in lighter environments where xarray falls back to another
        # installed backend. Do not catch PermissionError/OSError here; those
        # are handled by write_netcdf_atomic.
        if "netCDF4" not in str(exc) and "netcdf4" not in str(exc):
            raise
        ds.to_netcdf(path, mode="w")

def write_netcdf_atomic(ds: xr.Dataset, out_path: Path) -> Path:
    """Write a NetCDF robustly on Windows and return the actual path used.

    Safeguards:
    - uses a deterministic short target path when the requested path is too
      long for Windows/netCDF4;
    - writes through a deliberately short same-directory temporary file;
    - falls back to a unique short filename if the target is locked;
    - never lets cleanup of a missing temp file mask the real write outcome.
    """
    requested_path = Path(out_path)
    requested_path.parent.mkdir(parents=True, exist_ok=True)
    final_path = resolve_netcdf_cache_path(requested_path)
    final_path.parent.mkdir(parents=True, exist_ok=True)

    if final_path != requested_path:
        warnings.warn(
            "Requested NetCDF path is too long for reliable Windows/netCDF4 writes. "
            f"Using deterministic short path instead:\n  requested: {requested_path}\n  actual:    {final_path}",
            RuntimeWarning,
        )

    tmp_path = _short_temp_netcdf_path(final_path)
    ds_to_write = ds.load()
    _clear_xarray_file_cache()

    try:
        try:
            _write_dataset_to_netcdf(ds_to_write, tmp_path)
        except (PermissionError, OSError, FileNotFoundError) as first_error:
            fallback_path = _fallback_netcdf_path(requested_path, reason="writefail")
            try:
                _write_dataset_to_netcdf(ds_to_write, fallback_path)
            except Exception as fallback_error:
                raise RuntimeError(
                    "Failed to write NetCDF forecast even after switching to a short fallback filename. "
                    f"Requested path: {requested_path}. Short target path: {final_path}. "
                    f"Temporary path: {tmp_path}. Close open datasets/viewers and make sure the output directory is writable."
                ) from fallback_error
            warnings.warn(
                "Could not write the short temporary NetCDF file, so a unique fallback forecast was written instead: "
                f"{fallback_path}. Original write error was: {type(first_error).__name__}: {first_error}",
                RuntimeWarning,
            )
            return fallback_path

        _clear_xarray_file_cache()
        if not tmp_path.exists():
            fallback_path = _fallback_netcdf_path(requested_path, reason="missingtmp")
            _write_dataset_to_netcdf(ds_to_write, fallback_path)
            warnings.warn(
                "The temporary NetCDF file was not present after writing, so the forecast was written directly "
                f"to a unique fallback path: {fallback_path}",
                RuntimeWarning,
            )
            return fallback_path

        try:
            os.replace(tmp_path, final_path)
            return final_path
        except (PermissionError, OSError, FileNotFoundError) as replace_error:
            # FileNotFoundError/OSError is common on Windows when the destination
            # path is over MAX_PATH or a sync/indexing process races the replace.
            reason = "locked" if isinstance(replace_error, PermissionError) else "replacefail"
            fallback_path = _fallback_netcdf_path(requested_path, reason=reason)
            try:
                if tmp_path.exists():
                    os.replace(tmp_path, fallback_path)
                else:
                    _write_dataset_to_netcdf(ds_to_write, fallback_path)
            except Exception:
                _write_dataset_to_netcdf(ds_to_write, fallback_path)
            warnings.warn(
                "Could not move the temporary NetCDF into the requested target path, so a unique short forecast "
                f"was written instead: {fallback_path}. Replace error was: "
                f"{type(replace_error).__name__}: {replace_error}",
                RuntimeWarning,
            )
            return fallback_path
    finally:
        try:
            tmp_path.unlink(missing_ok=True)
        except PermissionError:
            pass
        except FileNotFoundError:
            pass

def target_burn_cap_value(cap: bool | str | float | None) -> float | None:
    """Resolve a final burned-fraction cap used after inverse transform only."""
    if cap is False or cap is None:
        return None
    if isinstance(cap, str):
        mode = cap.lower().strip()
        if mode in {"", "false", "none", "no", "off"}:
            return None
        lookup = {
            "raw_p99": "raw_y_p99",
            "raw_p999": "raw_y_p999",
            "raw_p9999": "raw_y_p9999",
            "raw_max": "raw_y_max",
            "train_p99": "raw_y_p99",
            "train_p999": "raw_y_p999",
            "train_p9999": "raw_y_p9999",
            "train_max": "raw_y_max",
        }
        if mode not in lookup:
            raise ValueError(f"Unknown final burned-fraction cap mode: {cap!r}")
        return float(target_transform_stats[lookup[mode]])
    return float(cap)


@torch.no_grad()
def ddim_sample(
    model: nn.Module,
    schedule: DiffusionSchedule,
    conditioning: torch.Tensor,
    shape: tuple[int, int, int, int],
    steps: int = DDIM_STEPS,
    eta: float = DDIM_ETA,
    clip_x0: bool | str | None = DDIM_CLIP_X0_EACH_STEP,
    x_T_scale: float = 1.0,
    final_x0_clip: bool | str | None = None,
) -> torch.Tensor:
    """Generate scaled target samples with DDIM.

    `clip_x0` clips predicted x0 at every reverse step. v7 sampler calibration
    usually sets this to False and instead applies `final_x0_clip` after the last
    reverse step. `x_T_scale` changes the standard deviation of the initial noise.
    """
    model.eval()
    bsz = shape[0]
    x = torch.randn(shape, device=conditioning.device) * float(x_T_scale)
    step_indices = torch.linspace(schedule.num_timesteps - 1, 0, steps, device=conditioning.device).long()
    step_indices = torch.unique_consecutive(step_indices)

    for i, t_scalar in enumerate(step_indices):
        t_int = int(t_scalar.item())
        t = torch.full((bsz,), t_int, device=conditioning.device, dtype=torch.long)
        eps = model(x, conditioning, t)
        alpha_bar_t = schedule.alphas_cumprod[t_int]
        sqrt_alpha_bar_t = torch.sqrt(alpha_bar_t)
        sqrt_one_minus_alpha_bar_t = torch.sqrt(1.0 - alpha_bar_t)
        pred_x0 = (x - sqrt_one_minus_alpha_bar_t * eps) / sqrt_alpha_bar_t
        pred_x0 = clip_predicted_x0_if_requested(pred_x0, clip_x0)

        if i == len(step_indices) - 1:
            alpha_bar_prev = torch.tensor(1.0, device=conditioning.device, dtype=x.dtype)
        else:
            t_prev = int(step_indices[i + 1].item())
            alpha_bar_prev = schedule.alphas_cumprod[t_prev]

        if eta > 0 and i < len(step_indices) - 1:
            sigma = eta * torch.sqrt(
                torch.clamp((1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t) * (1.0 - alpha_bar_t / alpha_bar_prev), min=0.0)
            )
            noise = torch.randn_like(x)
        else:
            sigma = torch.tensor(0.0, device=conditioning.device, dtype=x.dtype)
            noise = torch.zeros_like(x)

        direction = torch.sqrt(torch.clamp(1.0 - alpha_bar_prev - sigma ** 2, min=0.0)) * eps
        x = torch.sqrt(alpha_bar_prev) * pred_x0 + direction + sigma * noise

    x = clip_predicted_x0_if_requested(x, final_x0_clip)
    return x


def load_diffusion_model_for_sampling(
    checkpoint_path: Path = BEST_CHECKPOINT_PATH,
    use_ema: bool = USE_EMA_FOR_SAMPLING,
) -> tuple[nn.Module, dict]:
    """Load either raw or EMA checkpoint weights for sampling/debug comparison."""
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"No diffusion checkpoint found at {checkpoint_path}. Run training first.")
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    cfg = ckpt.get("config", config)
    loaded = ConditionalUNetDenoiser(
        cond_channels=int(cfg.get("cond_channels", cond_channels)),
        base_width=int(cfg.get("base_width", BASE_WIDTH)),
        time_dim=int(cfg.get("time_emb_dim", TIME_EMB_DIM)),
        dropout=float(cfg.get("dropout", DROPOUT)),
    ).to(DEVICE)
    loaded.load_state_dict(ckpt["model_state_dict"])
    if use_ema and "ema_state_dict" in ckpt:
        shadow = {name: value.to(DEVICE) for name, value in ckpt["ema_state_dict"].items()}
        with torch.no_grad():
            for name, param in loaded.named_parameters():
                if name in shadow:
                    param.data.copy_(shadow[name].data)
        print(f"Loaded EMA weights from {checkpoint_path} at epoch {ckpt.get('epoch')}.")
    else:
        print(f"Loaded raw model weights from {checkpoint_path} at epoch {ckpt.get('epoch')}.")
    loaded.eval()
    return loaded, cfg


def loader_for_sampling(dataset: WildfireDiffusionDataset) -> DataLoader:
    return DataLoader(dataset, batch_size=SAMPLE_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))


@torch.no_grad()
def generate_diffusion_forecast(
    model: nn.Module,
    schedule: DiffusionSchedule,
    dataset: WildfireDiffusionDataset,
    split_name: str,
    ensemble_size: int = ENSEMBLE_SIZE,
    ddim_steps: int = DDIM_STEPS,
    ddim_eta: float = DDIM_ETA,
    clip_x0: bool | str | None = DDIM_CLIP_X0_EACH_STEP,
    use_ema_weights: bool | None = None,
    model_label: str = "conditional_ddpm_unet",
    out_path: Path | None = None,
    regenerate: bool | None = None,
    x_T_scale: float = 1.0,
    final_x0_clip: bool | str | None = None,
    final_y_cap: bool | str | float | None = None,
) -> xr.DataArray:
    """Generate an ensemble forecast DataArray with dims (time, member, lat, lon)."""
    if regenerate is None:
        regenerate = REGENERATE_FORECASTS
    cache_path = resolve_netcdf_cache_path(out_path) if out_path is not None else None
    if cache_path is not None and cache_path.exists() and not regenerate:
        da = load_forecast_from_netcdf(cache_path)
        da.attrs["saved_path"] = str(cache_path)
        print(f"Loaded existing diffusion forecast: {cache_path}")
        return da.astype("float32")

    y_cap_value = target_burn_cap_value(final_y_cap)
    sample_loader = loader_for_sampling(dataset)
    members = []
    for member in tqdm(range(ensemble_size), desc=f"DDIM members {split_name}/{model_label}"):
        member_batches = []
        for batch in tqdm(sample_loader, leave=False, desc=f"member {member}"):
            conditioning = batch["conditioning"].to(DEVICE, non_blocking=True)
            shape = (conditioning.shape[0], 1, conditioning.shape[-2], conditioning.shape[-1])
            scaled_sample = ddim_sample(
                model=model,
                schedule=schedule,
                conditioning=conditioning,
                shape=shape,
                steps=ddim_steps,
                eta=ddim_eta,
                clip_x0=clip_x0,
                x_T_scale=x_T_scale,
                final_x0_clip=final_x0_clip,
            )
            y = diffusion_to_target_scale(scaled_sample).detach().cpu().numpy()[:, 0, :, :]
            if y_cap_value is not None:
                y = np.clip(y, 0.0, y_cap_value)
            member_batches.append(y.astype("float32"))
        members.append(np.concatenate(member_batches, axis=0))

    forecast_np = np.stack(members, axis=1)  # time, member, lat, lon
    forecast = xr.DataArray(
        forecast_np,
        dims=("time", "member", "lat", "lon"),
        coords={
            "time": dataset.times,
            "member": np.arange(ensemble_size, dtype="int64"),
            "lat": dataset.lat,
            "lon": dataset.lon,
        },
        name="forecast",
        attrs={
            "model": model_label,
            "checkpoint": str(VALIDATION_CHECKPOINT_PATH if "VALIDATION_CHECKPOINT_PATH" in globals() else BEST_CHECKPOINT_PATH),
            "uses_ema_weights": str(use_ema_weights),
            "ddpm_timesteps": schedule.num_timesteps,
            "ddim_steps": ddim_steps,
            "ddim_eta": ddim_eta,
            "clip_x0_each_step": str(clip_x0),
            "x_T_scale": float(x_T_scale),
            "final_x0_clip": str(final_x0_clip),
            "final_y_cap": str(final_y_cap),
            "final_y_cap_value": str(y_cap_value),
            "ensemble_size": ensemble_size,
            "target_mode": TARGET_MODE,
            "target_transform_stats": json.dumps(target_transform_stats),
        },
    ).where(land_mask)

    if out_path is not None:
        saved_path = write_netcdf_atomic(forecast.to_dataset(name="forecast"), out_path)
        forecast.attrs["saved_path"] = str(saved_path)
        print(f"Saved diffusion forecast: {saved_path}")
    return forecast.astype("float32")


In [ ]:
diffusion_forecasts = {}
sampler_debug_forecasts = {}
sampler_debug_records = []
sampler_debug_targets = {}
eval_splits = ["val"] + (["test"] if RUN_TEST_EVAL else [])
sampling_datasets = {"val": val_dataset, "test": test_dataset}

if RUN_VAL_SAMPLING:
    sampling_model, sampling_config = load_diffusion_model_for_sampling(BEST_CHECKPOINT_PATH, use_ema=USE_EMA_FOR_SAMPLING)
    sampling_schedule = DiffusionSchedule(int(sampling_config.get("ddpm_timesteps", DDPM_TIMESTEPS)), DEVICE)
    for split_name in eval_splits:
        out_path = FORECAST_DIR / f"{DIFFUSION_EXPERIMENT_ID}_ddim{DDIM_STEPS}_ens{ENSEMBLE_SIZE}_forecast_{split_name}.nc"
        diffusion_forecasts[(DIFFUSION_MODEL_NAME, split_name)] = generate_diffusion_forecast(
            model=sampling_model,
            schedule=sampling_schedule,
            dataset=sampling_datasets[split_name],
            split_name=split_name,
            ensemble_size=ENSEMBLE_SIZE,
            ddim_steps=DDIM_STEPS,
            ddim_eta=DDIM_ETA,
            clip_x0=DDIM_CLIP_X0_EACH_STEP,
            use_ema_weights=USE_EMA_FOR_SAMPLING,
            model_label=DIFFUSION_MODEL_NAME,
            out_path=out_path,
        )
else:
    print("RUN_VAL_SAMPLING=False: skipping standard diffusion forecast generation.")



def _xt_label(x: float) -> str:
    """Compact label for x_T_scale, e.g. 1.00 -> t100, 0.975 -> t0975."""
    value = int(round(float(x) * 1000))
    if value % 10 == 0:
        return f"t{value // 10:03d}"
    return f"t{value:04d}"


def _eta_label(eta: float) -> str:
    """Compact label for DDIM eta, e.g. 0.20 -> e02."""
    return f"e{int(round(float(eta) * 10)):02d}"


def make_validation_sampler_variants() -> list[dict]:
    """Final frozen sampler settings.

    The model is labelled `d10f` in outputs to keep tables readable, but it
    loads the validation-selected d10a checkpoint.
    """
    return [{
        "label": "final",
        "model_name": DIFFUSION_MODEL_NAME,
        "description": "locked d10a raw weights; x_T_scale=1.00; DDIM eta=0.50; final raw p999 cap",
        "use_ema": False,
        "x_T_scale": 1.0,
        "ddim_eta": 0.50,
        "clip_x0": False,
        "final_x0_clip": False,
        "final_y_cap": VALIDATION_FINAL_Y_CAP,
    }]

def _require_validation_checkpoint(path: Path) -> None:
    if not Path(path).exists():
        raise FileNotFoundError(
            "No full diffusion checkpoint was found for validation sampling.\n"
            f"Expected: {path}\n\n"
            "Options:\n"
            "  1. Keep RUN_TRAIN=True and run the training cells until a best checkpoint is written.\n"
            "  2. Set VALIDATION_CHECKPOINT_PATH_OVERRIDE to an existing compatible full-checkpoint .pt file.\n\n"
            "Do not use the tiny-overfit checkpoint for validation model selection."
        )


validation_sampler_records = []
if RUN_VALIDATION_SAMPLER_PROBE:
    _require_validation_checkpoint(VALIDATION_CHECKPOINT_PATH)
    validation_sampler_dir = FORECAST_DIR / f"sampler_{DIFFUSION_MODEL_NAME}"
    validation_sampler_dir.mkdir(parents=True, exist_ok=True)
    validation_variants = make_validation_sampler_variants()
    print(f"Running {len(validation_variants)} validation sampler variants from {VALIDATION_CHECKPOINT_PATH}")

    for variant in validation_variants:
        label = str(variant["label"])
        use_ema = bool(variant.get("use_ema", False))
        model_name = str(variant.get("model_name", f"{DIFFUSION_MODEL_NAME}_{label}"))
        sampling_model, sampling_config = load_diffusion_model_for_sampling(VALIDATION_CHECKPOINT_PATH, use_ema=use_ema)
        sampling_schedule = DiffusionSchedule(int(sampling_config.get("ddpm_timesteps", DDPM_TIMESTEPS)), DEVICE)
        for split_name in eval_splits:
            out_path = validation_sampler_dir / (
                f"{DIFFUSION_MODEL_NAME}_{label}_s{VALIDATION_SAMPLER_DDIM_STEPS}"
                f"_m{VALIDATION_SAMPLER_ENSEMBLE_SIZE}_{split_name}.nc"
            )
            forecast = generate_diffusion_forecast(
                model=sampling_model,
                schedule=sampling_schedule,
                dataset=sampling_datasets[split_name],
                split_name=split_name,
                ensemble_size=VALIDATION_SAMPLER_ENSEMBLE_SIZE,
                ddim_steps=VALIDATION_SAMPLER_DDIM_STEPS,
                ddim_eta=float(variant.get("ddim_eta", DDIM_ETA)),
                clip_x0=variant.get("clip_x0", False),
                use_ema_weights=use_ema,
                model_label=model_name,
                out_path=out_path,
                regenerate=VALIDATION_SAMPLER_REGENERATE,
                x_T_scale=float(variant.get("x_T_scale", 1.0)),
                final_x0_clip=variant.get("final_x0_clip", False),
                final_y_cap=variant.get("final_y_cap", False),
            )
            diffusion_forecasts[(model_name, split_name)] = forecast
            validation_sampler_records.append({
                "model": model_name,
                "split": split_name,
                "variant_label": label,
                "description": variant.get("description", ""),
                "forecast_path": str(forecast.attrs.get("saved_path", out_path)),
                "checkpoint_path": str(VALIDATION_CHECKPOINT_PATH),
                "uses_ema_weights": use_ema,
                "clip_x0": str(variant.get("clip_x0", False)),
                "x_T_scale": float(variant.get("x_T_scale", 1.0)),
                "final_x0_clip": str(variant.get("final_x0_clip", False)),
                "final_y_cap": str(variant.get("final_y_cap", False)),
                "final_y_cap_value": target_burn_cap_value(variant.get("final_y_cap", False)),
                "ddim_steps": VALIDATION_SAMPLER_DDIM_STEPS,
                "ddim_eta": float(variant.get("ddim_eta", DDIM_ETA)),
                "ensemble_size": VALIDATION_SAMPLER_ENSEMBLE_SIZE,
                "test_eval_enabled": RUN_TEST_EVAL,
            })
        del sampling_model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    validation_sampler_manifest_df = pd.DataFrame(validation_sampler_records)
    validation_sampler_manifest_df.to_csv(VALIDATION_SAMPLER_MANIFEST_PATH, index=False)
    print(f"Saved validation sampler manifest: {VALIDATION_SAMPLER_MANIFEST_PATH}")
    display(validation_sampler_manifest_df)
else:
    validation_sampler_manifest_df = pd.DataFrame()
    print("RUN_VALIDATION_SAMPLER_PROBE=False: skipping v10 validation sampler variants.")


class DatasetTimeSubset(Dataset):
    """A lightweight time-indexed view used for train-split sampler debugging."""

    def __init__(self, base: WildfireDiffusionDataset, indices: list[int] | np.ndarray, split_name: str):
        self.base = base
        self.indices = np.asarray(indices, dtype="int64")
        self.split_name = split_name
        self.channels = base.channels
        self.times = pd.DatetimeIndex(base.times[self.indices])
        self.lat = base.lat
        self.lon = base.lon

    def __len__(self) -> int:
        return int(len(self.indices))

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor | str]:
        return self.base[int(self.indices[idx])]


def select_debug_month_indices(target: xr.DataArray, n_months: int | None = SAMPLER_DEBUG_TRAIN_N_MONTHS) -> np.ndarray:
    """Pick a small, deterministic mix of high-burn and spread-out train months."""
    n_total = int(target.sizes["time"])
    if n_months is None or n_months >= n_total:
        return np.arange(n_total, dtype="int64")

    n_months = max(1, int(n_months))
    monthly_total = target.where(land_mask).fillna(0).sum(("lat", "lon")).values.astype("float64")
    all_ranked = list(np.argsort(monthly_total)[::-1])
    fire_mask = np.isin(pd.DatetimeIndex(target.time.values).month, FIRE_SEASON_MONTHS)
    fire_indices = np.where(fire_mask)[0]
    fire_ranked = list(fire_indices[np.argsort(monthly_total[fire_indices])[::-1]]) if len(fire_indices) else []
    spaced = list(np.linspace(0, n_total - 1, n_months, dtype="int64"))

    selected: list[int] = []
    for pool in [all_ranked[: max(2, n_months // 3)], fire_ranked[: max(2, n_months // 3)], spaced]:
        for idx in pool:
            idx = int(idx)
            if idx not in selected:
                selected.append(idx)
            if len(selected) >= n_months:
                break
        if len(selected) >= n_months:
            break
    return np.array(sorted(selected), dtype="int64")


def _debug_model_name(label: str) -> str:
    return f"diffusion_debug_{label}"


if RUN_SAMPLER_DEBUG_CHECK and not (RUN_OVERFIT_DEBUG and STOP_AFTER_OVERFIT_DEBUG):
    debug_dir = FORECAST_DIR / "sampler_debug"
    debug_dir.mkdir(parents=True, exist_ok=True)
    train_debug_indices = select_debug_month_indices(targets["train"], SAMPLER_DEBUG_TRAIN_N_MONTHS)
    train_debug_dataset = DatasetTimeSubset(train_dataset, train_debug_indices, split_name="train_debug")
    targets["train_debug"] = targets["train"].isel(time=train_debug_indices).assign_coords(
        split=("time", np.repeat("train_debug", len(train_debug_indices)))
    )
    sampler_debug_targets = {"val": targets["val"], "train_debug": targets["train_debug"]}
    debug_selected_months_path = METRIC_DIR / "diffusion_sampler_debug_selected_train_months.csv"
    pd.DataFrame({
        "split": "train_debug",
        "time_index": train_debug_indices,
        "time": [str(t.date()) for t in train_debug_dataset.times],
        "land_burn_sum": targets["train_debug"].where(land_mask).fillna(0).sum(("lat", "lon")).values,
    }).to_csv(debug_selected_months_path, index=False)
    print(f"Saved sampler-debug selected train months: {debug_selected_months_path}")

    debug_sampling_datasets = {"val": val_dataset, "train_debug": train_debug_dataset}
    for variant in SAMPLER_DEBUG_VARIANTS:
        label = str(variant["label"])
        use_ema = bool(variant.get("use_ema", True))
        clip_x0 = bool(variant.get("clip_x0", True))
        steps = int(variant.get("steps", SAMPLER_DEBUG_DDIM_STEPS))
        model_name = _debug_model_name(label)

        debug_model, debug_config = load_diffusion_model_for_sampling(BEST_CHECKPOINT_PATH, use_ema=use_ema)
        debug_schedule = DiffusionSchedule(int(debug_config.get("ddpm_timesteps", DDPM_TIMESTEPS)), DEVICE)
        for split_name, dataset in debug_sampling_datasets.items():
            out_path = debug_dir / f"{model_name}_ens{SAMPLER_DEBUG_ENSEMBLE_SIZE}_forecast_{split_name}.nc"
            forecast = generate_diffusion_forecast(
                model=debug_model,
                schedule=debug_schedule,
                dataset=dataset,
                split_name=split_name,
                ensemble_size=SAMPLER_DEBUG_ENSEMBLE_SIZE,
                ddim_steps=steps,
                ddim_eta=DDIM_ETA,
                clip_x0=clip_x0,
                use_ema_weights=use_ema,
                model_label=model_name,
                out_path=out_path,
                regenerate=SAMPLER_DEBUG_REGENERATE,
            )
            sampler_debug_forecasts[(model_name, split_name)] = forecast
            sampler_debug_records.append({
                "model": model_name,
                "split": split_name,
                "forecast_path": str(out_path),
                "use_ema": use_ema,
                "clip_x0": clip_x0,
                "ddim_steps": steps,
                "ddim_eta": float(variant.get("ddim_eta", DDIM_ETA)),
                "ensemble_size": SAMPLER_DEBUG_ENSEMBLE_SIZE,
            })
            if split_name == "val":
                diffusion_forecasts[(model_name, "val")] = forecast
else:
    print("RUN_SAMPLER_DEBUG_CHECK=False: skipping sampler/debug forecast variants.")


## 9. Reference forecasts: climatology, persistence, deterministic U-Net

Climatology and persistence are regenerated here to make this notebook self-contained. The deterministic U-Net forecast is loaded from the deterministic notebook outputs. If the deterministic forecast is missing, run the final deterministic U-Net notebook first, or copy the relevant forecast NetCDF into `data/processed/forecasts/deterministic_unet/`.

In [ ]:
def add_member_dim(da: xr.DataArray, member_label: int = 0, name: str = "forecast") -> xr.DataArray:
    if "member" in da.dims:
        out = da
    else:
        out = da.expand_dims(member=[member_label])
    desired_order = [dim for dim in ["time", "member", "lat", "lon"] if dim in out.dims]
    return out.transpose(*desired_order).rename(name).astype("float32")


def fit_monthly_climatology(train_target: xr.DataArray) -> xr.DataArray:
    """Train-only per-pixel monthly mean burned fraction."""
    clim = train_target.groupby("time.month").mean("time", skipna=True).astype("float32")
    clim = clim.rename("climatology_monthly_mean")
    clim.attrs.update({"description": "Train-only per-pixel monthly burned-fraction climatology.", "fit_split": "train"})
    return clim


def predict_monthly_climatology(climatology: xr.DataArray, target_template: xr.DataArray) -> xr.DataArray:
    pieces = []
    for timestamp in pd.DatetimeIndex(target_template.time.values):
        pred = climatology.sel(month=timestamp.month).drop_vars("month", errors="ignore")
        pred = pred.assign_coords(time=timestamp).expand_dims("time")
        pieces.append(pred)
    forecast = xr.concat(pieces, dim="time").assign_coords(time=target_template.time)
    forecast = forecast.where(land_mask)
    return add_member_dim(forecast, name="forecast")


def predict_persistence(full_target_record: xr.DataArray, target_template: xr.DataArray) -> xr.DataArray:
    """Previous observed target month copied forward as a next-month forecast."""
    persistence_all = full_target_record.sortby("time").shift(time=1).rename("persistence_forecast")
    forecast = persistence_all.sel(time=target_template.time).where(land_mask)
    return add_member_dim(forecast, name="forecast")


def load_forecast_nc(path: Path) -> xr.DataArray:
    """Load a forecast NetCDF fully into memory and close the file handle."""
    with xr.open_dataset(path) as ds:
        da = ds["forecast"] if "forecast" in ds.data_vars else ds[list(ds.data_vars)[0]]
        da = da.sortby("lat").load()
    return da.astype("float32")


def find_deterministic_forecast(split_name: str) -> Path | None:
    candidates = [
        DETERMINISTIC_FORECAST_DIR / f"{SELECTED_DETERMINISTIC_MODEL}_forecast_{split_name}.nc",
        DATA_DIR / "forecasts" / "deterministic_unet" / f"{SELECTED_DETERMINISTIC_MODEL}_forecast_{split_name}.nc",
        DATA_DIR / "forecasts" / f"{SELECTED_DETERMINISTIC_MODEL}_forecast_{split_name}.nc",
    ]
    for path in candidates:
        if path.exists():
            return path
    return None


climatology = fit_monthly_climatology(targets["train"])
climatology_path = FORECAST_DIR / "climatology_monthly_mean_train_only.nc"
climatology_saved_path = write_netcdf_atomic(climatology.to_dataset(name="climatology_monthly_mean"), climatology_path)
print(f"Saved climatology fit: {climatology_saved_path}")

reference_forecasts = {}
for split_name in eval_splits:
    reference_forecasts[("climatology", split_name)] = predict_monthly_climatology(climatology, targets[split_name])
    reference_forecasts[("persistence", split_name)] = predict_persistence(target_all, targets[split_name])

    det_path = find_deterministic_forecast(split_name)
    if det_path is not None:
        reference_forecasts[("deterministic_unet_mild2_tail2", split_name)] = load_forecast_nc(det_path)
        print(f"Loaded deterministic U-Net forecast for {split_name}: {det_path}")
    else:
        print(f"WARNING: deterministic U-Net forecast for {split_name} not found under {DETERMINISTIC_FORECAST_DIR}")

for (model_name, split_name), forecast in reference_forecasts.items():
    out_path = FORECAST_DIR / f"{model_name}_forecast_{split_name}.nc"
    saved_path = write_netcdf_atomic(forecast.to_dataset(name="forecast"), out_path)
    forecast.attrs["saved_path"] = str(saved_path)
    print(f"Saved/linked {model_name}/{split_name}: {saved_path}")

all_forecasts = {}
all_forecasts.update(diffusion_forecasts)
all_forecasts.update(reference_forecasts)
print("Forecasts available for scoring:")
for key, forecast in all_forecasts.items():
    print(f"  {key}: {tuple(forecast.shape)}")

## 10. Evaluation: CRPS, deterministic metrics, monthly scores, and reliability

For one-member deterministic baselines, ensemble CRPS reduces to MAE. For the diffusion ensemble, CRPS measures the full predictive distribution and is the primary validation metric.

In [ ]:
def ensemble_crps_sorted(
    forecast: xr.DataArray,
    observation: xr.DataArray,
    member_dim: str = "member",
) -> xr.DataArray:
    """Efficient empirical CRPS using the sorted-ensemble identity.

    Ocean/excluded cells are all-NaN after masking. This implementation marks those
    cells as NaN without calling nanmean on empty member slices, avoiding the repeated
    RuntimeWarning seen in the raw-target debug run.
    """
    forecast, observation = xr.align(forecast, observation, join="inner")
    forecast = forecast.transpose("time", member_dim, "lat", "lon")
    observation = observation.transpose("time", "lat", "lon")
    f = np.asarray(forecast.values, dtype="float64")
    y = np.asarray(observation.values, dtype="float64")
    m = f.shape[1]
    full_valid = np.isfinite(y) & np.all(np.isfinite(f), axis=1)

    mae_term = np.mean(np.abs(f - y[:, None, :, :]), axis=1)
    sorted_f = np.sort(f, axis=1)
    coeff = (2 * np.arange(1, m + 1) - m - 1).astype("float64")
    spread_term = np.sum(sorted_f * coeff[None, :, None, None], axis=1) / (m * m)
    crps = mae_term - spread_term
    crps = np.where(full_valid, crps, np.nan)
    return xr.DataArray(
        crps.astype("float32"),
        dims=("time", "lat", "lon"),
        coords={"time": observation.time, "lat": observation.lat, "lon": observation.lon},
        name="crps",
    )


def weighted_masked_mean(da: xr.DataArray, weights: xr.DataArray, condition: xr.DataArray | None = None) -> float:
    valid = xr.apply_ufunc(np.isfinite, da)
    if condition is not None:
        valid = valid & condition
    w = weights.broadcast_like(da).where(valid)
    denom = w.sum(skipna=True)
    if not np.isfinite(float(denom)) or float(denom) <= 0:
        return np.nan
    return float(((da * w).where(valid).sum(skipna=True) / denom).values)


def valid_cell_month_count(da: xr.DataArray, weights: xr.DataArray, condition: xr.DataArray | None = None) -> int:
    valid = xr.apply_ufunc(np.isfinite, da) & xr.apply_ufunc(np.isfinite, weights.broadcast_like(da)) & (weights.broadcast_like(da) > 0)
    if condition is not None:
        valid = valid & condition
    return int(valid.sum().values)


def weighted_correlation(x: xr.DataArray, y: xr.DataArray, weights: xr.DataArray, condition: xr.DataArray | None = None) -> float:
    x, y = xr.align(x, y, join="inner")
    valid = xr.apply_ufunc(np.isfinite, x) & xr.apply_ufunc(np.isfinite, y)
    if condition is not None:
        valid = valid & condition
    w = weights.broadcast_like(x).where(valid)
    denom = w.sum(skipna=True)
    if not np.isfinite(float(denom)) or float(denom) <= 0:
        return np.nan
    x_mean = (x * w).sum(skipna=True) / denom
    y_mean = (y * w).sum(skipna=True) / denom
    x_anom = x - x_mean
    y_anom = y - y_mean
    cov = float(((x_anom * y_anom * w).sum(skipna=True) / denom).values)
    x_var = float((((x_anom ** 2) * w).sum(skipna=True) / denom).values)
    y_var = float((((y_anom ** 2) * w).sum(skipna=True) / denom).values)
    if x_var <= 0 or y_var <= 0:
        return np.nan
    return cov / math.sqrt(x_var * y_var)


def make_subset_conditions(observation: xr.DataArray, train_tail_threshold: float = train_tail_threshold) -> dict[str, xr.DataArray]:
    month = observation["time"].dt.month
    month_3d = xr.DataArray(
        np.broadcast_to(month.values[:, None, None], observation.shape),
        dims=observation.dims,
        coords=observation.coords,
    )
    land_3d = land_mask.broadcast_like(observation).astype(bool)
    return {
        "all": land_3d,
        "fire_season_Jul-Oct": land_3d & month_3d.isin(FIRE_SEASON_MONTHS),
        "august": land_3d & (month_3d == 8),
        "active_burned_gt_1e-6": land_3d & (observation > BURN_THRESHOLD),
        "top_1pct_train_tail": land_3d & (observation >= train_tail_threshold),
    }


def evaluate_forecast(
    forecast_ens: xr.DataArray,
    observation: xr.DataArray,
    weights: xr.DataArray,
    model_name: str,
    split_name: str,
    member_dim: str = "member",
) -> pd.DataFrame:
    forecast_ens, observation = xr.align(forecast_ens, observation, join="inner")
    forecast_mean = forecast_ens.mean(member_dim, skipna=True).rename("forecast_mean")
    error = (forecast_mean - observation).rename("error")
    crps = ensemble_crps_sorted(forecast_ens, observation, member_dim=member_dim)
    spread = forecast_ens.std(member_dim, skipna=True).rename("ensemble_spread") if forecast_ens.sizes.get(member_dim, 1) > 1 else xr.zeros_like(observation)
    conditions = make_subset_conditions(observation)

    rows = []
    for subset_name, condition in conditions.items():
        mse = weighted_masked_mean(error ** 2, weights, condition)
        rows.append({
            "model": model_name,
            "split": split_name,
            "subset": subset_name,
            "n_members": int(forecast_ens.sizes.get(member_dim, 1)),
            "mse": mse,
            "rmse": math.sqrt(mse) if np.isfinite(mse) else np.nan,
            "mae": weighted_masked_mean(np.abs(error), weights, condition),
            "crps": weighted_masked_mean(crps, weights, condition),
            "bias": weighted_masked_mean(error, weights, condition),
            "correlation": weighted_correlation(forecast_mean, observation, weights, condition),
            "obs_mean": weighted_masked_mean(observation, weights, condition),
            "forecast_mean": weighted_masked_mean(forecast_mean, weights, condition),
            "ensemble_spread_mean": weighted_masked_mean(spread, weights, condition),
            "n_cell_months": valid_cell_month_count(error, weights, condition),
        })
    return pd.DataFrame(rows)


def monthly_score_table(
    forecast_ens: xr.DataArray,
    observation: xr.DataArray,
    weights: xr.DataArray,
    model_name: str,
    split_name: str,
    member_dim: str = "member",
) -> pd.DataFrame:
    forecast_ens, observation = xr.align(forecast_ens, observation, join="inner")
    forecast_mean = forecast_ens.mean(member_dim, skipna=True)
    error = forecast_mean - observation
    crps = ensemble_crps_sorted(forecast_ens, observation, member_dim=member_dim)
    rows = []
    for month in range(1, 13):
        condition = land_mask.broadcast_like(observation) & (observation["time"].dt.month == month)
        mse = weighted_masked_mean(error ** 2, weights, condition)
        rows.append({
            "model": model_name,
            "split": split_name,
            "month": month,
            "mse": mse,
            "rmse": math.sqrt(mse) if np.isfinite(mse) else np.nan,
            "mae": weighted_masked_mean(np.abs(error), weights, condition),
            "crps": weighted_masked_mean(crps, weights, condition),
            "bias": weighted_masked_mean(error, weights, condition),
            "correlation": weighted_correlation(forecast_mean, observation, weights, condition),
            "n_cell_months": valid_cell_month_count(error, weights, condition),
        })
    return pd.DataFrame(rows)


def reliability_table(
    forecast_ens: xr.DataArray,
    observation: xr.DataArray,
    threshold: float,
    event_name: str,
    weights: xr.DataArray,
    model_name: str,
    split_name: str,
    member_dim: str = "member",
    n_bins: int = 10,
) -> pd.DataFrame:
    forecast_ens, observation = xr.align(forecast_ens, observation, join="inner")
    prob = (forecast_ens >= threshold).mean(member_dim, skipna=True).rename("forecast_probability")
    event = (observation >= threshold).astype("float32").rename("observed_event")
    land_3d = land_mask.broadcast_like(observation).astype(bool)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for i in range(n_bins):
        left, right = float(bin_edges[i]), float(bin_edges[i + 1])
        if i == n_bins - 1:
            condition = land_3d & (prob >= left) & (prob <= right)
        else:
            condition = land_3d & (prob >= left) & (prob < right)
        rows.append({
            "model": model_name,
            "split": split_name,
            "event": event_name,
            "threshold": threshold,
            "bin_left": left,
            "bin_right": right,
            "bin_mid": 0.5 * (left + right),
            "forecast_probability_mean": weighted_masked_mean(prob, weights, condition),
            "observed_frequency": weighted_masked_mean(event, weights, condition),
            "n_cell_months": valid_cell_month_count(event, weights, condition),
        })
    return pd.DataFrame(rows)



def forecast_distribution_table(
    forecast_ens: xr.DataArray,
    observation: xr.DataArray,
    weights: xr.DataArray,
    model_name: str,
    split_name: str,
    member_dim: str = "member",
) -> pd.DataFrame:
    """Compact distribution diagnostics for the ensemble mean and ensemble spread."""
    forecast_ens, observation = xr.align(forecast_ens, observation, join="inner")
    forecast_mean = forecast_ens.mean(member_dim, skipna=True)
    forecast_spread = forecast_ens.std(member_dim, skipna=True) if forecast_ens.sizes.get(member_dim, 1) > 1 else xr.zeros_like(observation)
    conditions = make_subset_conditions(observation)
    rows = []
    for subset_name, condition in conditions.items():
        vals = forecast_mean.where(condition).values.astype("float64")
        obs_vals = observation.where(condition).values.astype("float64")
        spread_vals = forecast_spread.where(condition).values.astype("float64")
        valid = np.isfinite(vals) & np.isfinite(obs_vals)
        if not valid.any():
            continue
        rows.append({
            "model": model_name,
            "split": split_name,
            "subset": subset_name,
            "n_cell_months": int(valid.sum()),
            "obs_mean_unweighted": float(np.nanmean(obs_vals[valid])),
            "forecast_mean_unweighted": float(np.nanmean(vals[valid])),
            "forecast_median_unweighted": float(np.nanmedian(vals[valid])),
            "forecast_p01_unweighted": float(np.nanpercentile(vals[valid], 1)),
            "forecast_p05_unweighted": float(np.nanpercentile(vals[valid], 5)),
            "forecast_p50_unweighted": float(np.nanpercentile(vals[valid], 50)),
            "forecast_p95_unweighted": float(np.nanpercentile(vals[valid], 95)),
            "forecast_p99_unweighted": float(np.nanpercentile(vals[valid], 99)),
            "spread_mean_unweighted": float(np.nanmean(spread_vals[valid])),
            "weighted_obs_mean": weighted_masked_mean(observation, weights, condition),
            "weighted_forecast_mean": weighted_masked_mean(forecast_mean, weights, condition),
            "weighted_spread_mean": weighted_masked_mean(forecast_spread, weights, condition),
        })
    return pd.DataFrame(rows)


def add_reference_skill_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for metric in ["rmse", "crps"]:
        for reference_model in ["climatology", "persistence", "deterministic_unet_mild2_tail2"]:
            ref = out[out["model"] == reference_model].set_index(["split", "subset"])[metric]
            col = f"{metric}_skill_vs_{reference_model}"
            out[col] = out.apply(
                lambda row: 1.0 - row[metric] / ref.loc[(row["split"], row["subset"])]
                if (row["split"], row["subset"]) in ref.index
                and np.isfinite(ref.loc[(row["split"], row["subset"])])
                and ref.loc[(row["split"], row["subset"])] > 0
                else np.nan,
                axis=1,
            )
    return out

## 10A. Post-sampling gate/scale calibration diagnostics

These diagnostics are validation-only and do not write new forecast NetCDFs. They take selected sparse sampler forecasts and test simple occurrence gates plus positive-only amplitude scaling. The goal is to see whether active/tail underprediction can be improved without recreating the `x_T_scale=0.75` false-positive haze.


In [ ]:
FALSE_POSITIVE_THRESHOLDS = (1e-4, 5e-4, 1e-3)


def zero_cell_false_positive_table(
    forecast_ens: xr.DataArray,
    observation: xr.DataArray,
    weights: xr.DataArray,
    model_name: str,
    split_name: str,
    member_dim: str = "member",
    thresholds: tuple[float, ...] = FALSE_POSITIVE_THRESHOLDS,
) -> pd.DataFrame:
    """Zero-cell false-positive diagnostics using the ensemble mean."""
    forecast_ens, observation = xr.align(forecast_ens, observation, join="inner")
    forecast_mean = forecast_ens.mean(member_dim, skipna=True)
    zero_condition = land_mask.broadcast_like(observation) & (observation <= BURN_THRESHOLD)
    values = forecast_mean.where(zero_condition).values.astype("float64")
    w = weights.broadcast_like(observation).where(zero_condition).values.astype("float64")
    valid = np.isfinite(values) & np.isfinite(w) & (w > 0)
    if not valid.any():
        return pd.DataFrame([{"model": model_name, "split": split_name, "n_zero_cell_months": 0}])

    vals = values[valid]
    ww = w[valid]
    row = {
        "model": model_name,
        "split": split_name,
        "n_zero_cell_months": int(valid.sum()),
        "zero_forecast_mean": float(np.average(vals, weights=ww)),
        "zero_forecast_median": float(np.nanmedian(vals)),
        "zero_forecast_p95": float(np.nanpercentile(vals, 95)),
        "zero_forecast_p99": float(np.nanpercentile(vals, 99)),
    }
    denom = ww.sum()
    for threshold in thresholds:
        row[f"zero_frac_pred_gt_{threshold:g}"] = float(ww[vals > threshold].sum() / denom) if denom > 0 else np.nan
    return pd.DataFrame([row])


def _threshold_label(value: float) -> str:
    if float(value) == 0.0:
        return "g0"
    return f"g{value:.0e}".replace("-", "m").replace("+", "").replace("e0", "e")


def _scale_label(value: float) -> str:
    return f"s{str(float(value)).replace('.', 'p')}"


def calibrate_forecast_members(
    forecast_ens: xr.DataArray,
    zero_below: float = 0.0,
    positive_scale: float = 1.0,
    final_y_cap: bool | str | float | None = POSTCAL_FINAL_Y_CAP,
) -> xr.DataArray:
    """Apply a simple member-wise occurrence gate and positive-only amplitude scale."""
    calibrated = forecast_ens
    zero_below = float(zero_below)
    positive_scale = float(positive_scale)
    if zero_below > 0:
        calibrated = calibrated.where(calibrated >= zero_below, 0.0)
    if positive_scale != 1.0:
        calibrated = xr.where(calibrated > 0.0, calibrated * positive_scale, calibrated)
    cap_value = target_burn_cap_value(final_y_cap)
    if cap_value is not None:
        calibrated = calibrated.clip(min=0.0, max=float(cap_value))
    else:
        calibrated = calibrated.clip(min=0.0)
    calibrated = calibrated.where(land_mask).astype("float32")
    calibrated.attrs.update(forecast_ens.attrs)
    calibrated.attrs.update({
        "postcal_zero_below": zero_below,
        "postcal_positive_scale": positive_scale,
        "postcal_final_y_cap": str(final_y_cap),
        "postcal_final_y_cap_value": str(cap_value),
    })
    return calibrated


def postcal_summary_table(metrics_df: pd.DataFrame, zero_fp_df: pd.DataFrame, records_df: pd.DataFrame) -> pd.DataFrame:
    """One-row-per-postcal-variant summary focused on headline subsets and zero-cell haze."""
    if metrics_df.empty:
        return pd.DataFrame()
    headline = metrics_df[(metrics_df["split"] == "val") & (metrics_df["subset"].isin(HEADLINE_SUBSETS))].copy()
    keep = ["model", "subset", "crps", "rmse", "mae", "bias", "correlation", "obs_mean", "forecast_mean", "ensemble_spread_mean"]
    headline = headline.loc[:, [c for c in keep if c in headline.columns]]
    pieces = []
    for subset in HEADLINE_SUBSETS:
        sub = headline[headline["subset"] == subset].drop(columns=["subset"], errors="ignore").copy()
        sub = sub.rename(columns={c: f"{subset}__{c}" for c in sub.columns if c != "model"})
        pieces.append(sub)
    summary = pieces[0] if pieces else pd.DataFrame()
    for piece in pieces[1:]:
        summary = summary.merge(piece, on="model", how="outer")
    if not zero_fp_df.empty:
        summary = summary.merge(zero_fp_df[zero_fp_df["split"] == "val"].drop(columns=["split"], errors="ignore"), on="model", how="left")
    if not records_df.empty:
        summary = summary.merge(records_df.drop(columns=["split"], errors="ignore"), on="model", how="left")
    sort_cols = [c for c in ["all__crps", "zero_forecast_mean"] if c in summary.columns]
    if sort_cols:
        summary = summary.sort_values(sort_cols)
    return summary


postcal_records = []
postcal_metrics_rows = []
postcal_zero_fp_rows = []
postcal_distribution_rows = []

if RUN_POST_SAMPLING_CALIBRATION_GRID:
    available_val_models = {model for (model, split) in all_forecasts if split == "val"}
    base_models = [model for model in POSTCAL_BASE_MODELS if model in available_val_models]
    missing_base_models = [model for model in POSTCAL_BASE_MODELS if model not in available_val_models]
    if missing_base_models:
        print(f"Post-calibration skipped missing base models: {missing_base_models}")
    if not base_models:
        print("No configured post-calibration base forecasts were found. Skipping post-calibration grid.")

    for base_model in base_models:
        base_forecast = all_forecasts[(base_model, "val")]
        for zero_below in POSTCAL_ZERO_BELOW_THRESHOLDS:
            for positive_scale in POSTCAL_POSITIVE_SCALES:
                # The identity row is still useful: it makes postcal CSVs self-contained.
                label = f"{base_model}_{_threshold_label(zero_below)}_{_scale_label(positive_scale)}"
                calibrated = calibrate_forecast_members(
                    base_forecast,
                    zero_below=float(zero_below),
                    positive_scale=float(positive_scale),
                    final_y_cap=POSTCAL_FINAL_Y_CAP,
                )
                postcal_records.append({
                    "model": label,
                    "split": "val",
                    "base_model": base_model,
                    "zero_below": float(zero_below),
                    "positive_scale": float(positive_scale),
                    "final_y_cap": str(POSTCAL_FINAL_Y_CAP),
                    "final_y_cap_value": target_burn_cap_value(POSTCAL_FINAL_Y_CAP),
                })
                postcal_metrics_rows.append(evaluate_forecast(calibrated, targets["val"], area_weights, label, "val"))
                postcal_zero_fp_rows.append(zero_cell_false_positive_table(calibrated, targets["val"], area_weights, label, "val"))
                postcal_distribution_rows.append(forecast_distribution_table(calibrated, targets["val"], area_weights, label, "val"))
                del calibrated

postcal_records_df = pd.DataFrame(postcal_records)
postcal_metrics_df = pd.concat(postcal_metrics_rows, ignore_index=True) if postcal_metrics_rows else pd.DataFrame()
postcal_zero_fp_df = pd.concat(postcal_zero_fp_rows, ignore_index=True) if postcal_zero_fp_rows else pd.DataFrame()
postcal_distribution_df = pd.concat(postcal_distribution_rows, ignore_index=True) if postcal_distribution_rows else pd.DataFrame()
postcal_summary_df = postcal_summary_table(postcal_metrics_df, postcal_zero_fp_df, postcal_records_df)

postcal_manifest_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_postcal_manifest.csv"
postcal_metrics_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_postcal_metrics.csv"
postcal_zero_fp_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_postcal_zero_fp.csv"
postcal_distribution_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_postcal_distribution.csv"
postcal_summary_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_postcal_summary.csv"

postcal_records_df.to_csv(postcal_manifest_path, index=False)
postcal_metrics_df.to_csv(postcal_metrics_path, index=False)
postcal_zero_fp_df.to_csv(postcal_zero_fp_path, index=False)
postcal_distribution_df.to_csv(postcal_distribution_path, index=False)
postcal_summary_df.to_csv(postcal_summary_path, index=False)

print(f"Saved post-calibration manifest:     {postcal_manifest_path}")
print(f"Saved post-calibration metrics:      {postcal_metrics_path}")
print(f"Saved post-calibration zero-cell FP: {postcal_zero_fp_path}")
print(f"Saved post-calibration distribution: {postcal_distribution_path}")
print(f"Saved post-calibration summary:      {postcal_summary_path}")
if not postcal_summary_df.empty:
    display_cols = [
        "model", "base_model", "zero_below", "positive_scale", "all__crps", "all__rmse",
        "active_burned_gt_1e-6__crps", "top_1pct_train_tail__crps",
        "zero_forecast_mean", "zero_frac_pred_gt_0.001",
    ]
    display(postcal_summary_df.loc[:, [c for c in display_cols if c in postcal_summary_df.columns]].head(25))


In [ ]:
metrics = []
monthly_metrics = []
reliability_rows = []
forecast_distribution_rows = []

for (model_name, split_name), forecast in all_forecasts.items():
    if split_name not in eval_splits:
        continue
    obs = targets[split_name]
    metrics.append(evaluate_forecast(forecast, obs, area_weights, model_name, split_name))
    monthly_metrics.append(monthly_score_table(forecast, obs, area_weights, model_name, split_name))
    forecast_distribution_rows.append(forecast_distribution_table(forecast, obs, area_weights, model_name, split_name))
    for event_name, threshold in {
        "active_gt_1e-6": BURN_THRESHOLD,
        "top_1pct_train_tail": train_tail_threshold,
    }.items():
        reliability_rows.append(reliability_table(forecast, obs, threshold, event_name, area_weights, model_name, split_name))

metrics_df = pd.concat(metrics, ignore_index=True) if metrics else pd.DataFrame()
if not metrics_df.empty:
    metrics_df = add_reference_skill_columns(metrics_df)
monthly_metrics_df = pd.concat(monthly_metrics, ignore_index=True) if monthly_metrics else pd.DataFrame()
reliability_df = pd.concat(reliability_rows, ignore_index=True) if reliability_rows else pd.DataFrame()
forecast_distribution_df = pd.concat(forecast_distribution_rows, ignore_index=True) if forecast_distribution_rows else pd.DataFrame()

metrics_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_eval_metrics.csv"
monthly_metrics_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_eval_monthly.csv"
reliability_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_eval_reliability.csv"
forecast_distribution_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_eval_distribution.csv"
metrics_df.to_csv(metrics_path, index=False)
monthly_metrics_df.to_csv(monthly_metrics_path, index=False)
reliability_df.to_csv(reliability_path, index=False)
forecast_distribution_df.to_csv(forecast_distribution_path, index=False)
print(f"Saved metrics:               {metrics_path}")
print(f"Saved monthly metrics:       {monthly_metrics_path}")
print(f"Saved reliability:           {reliability_path}")
print(f"Saved forecast distribution: {forecast_distribution_path}")

if not metrics_df.empty:
    display(
        metrics_df.query("subset in @HEADLINE_SUBSETS")
        .sort_values(["subset", "crps"])
        .loc[:, [
            "model", "split", "subset", "n_members", "crps", "rmse", "mae", "bias", "correlation",
            "obs_mean", "forecast_mean", "ensemble_spread_mean",
            "crps_skill_vs_climatology", "crps_skill_vs_persistence", "crps_skill_vs_deterministic_unet_mild2_tail2",
        ]]
    )

## 11. False-positive diagnostics and qualitative validation maps

These diagnostics are deliberately copied from the deterministic baseline workflow. They catch the failure mode where a model improves active/tail scores by raising a broad low-amplitude fire haze over cells that did not burn.

In [ ]:
FALSE_POSITIVE_THRESHOLDS = (1e-4, 5e-4, 1e-3)


def zero_cell_false_positive_table(
    forecast_ens: xr.DataArray,
    observation: xr.DataArray,
    weights: xr.DataArray,
    model_name: str,
    split_name: str,
    member_dim: str = "member",
    thresholds: tuple[float, ...] = FALSE_POSITIVE_THRESHOLDS,
) -> pd.DataFrame:
    forecast_ens, observation = xr.align(forecast_ens, observation, join="inner")
    forecast_mean = forecast_ens.mean(member_dim, skipna=True)
    zero_condition = land_mask.broadcast_like(observation) & (observation <= BURN_THRESHOLD)
    values = forecast_mean.where(zero_condition).values.astype("float64")
    w = weights.broadcast_like(observation).where(zero_condition).values.astype("float64")
    valid = np.isfinite(values) & np.isfinite(w) & (w > 0)
    if not valid.any():
        return pd.DataFrame([{"model": model_name, "split": split_name, "n_zero_cell_months": 0}])

    vals = values[valid]
    ww = w[valid]
    row = {
        "model": model_name,
        "split": split_name,
        "n_zero_cell_months": int(valid.sum()),
        "zero_forecast_mean": float(np.average(vals, weights=ww)),
        "zero_forecast_median": float(np.nanmedian(vals)),
        "zero_forecast_p95": float(np.nanpercentile(vals, 95)),
        "zero_forecast_p99": float(np.nanpercentile(vals, 99)),
    }
    denom = ww.sum()
    for threshold in thresholds:
        row[f"zero_frac_pred_gt_{threshold:g}"] = float(ww[vals > threshold].sum() / denom) if denom > 0 else np.nan
    return pd.DataFrame([row])


zero_fp_rows = []
for (model_name, split_name), forecast in all_forecasts.items():
    if split_name not in eval_splits:
        continue
    zero_fp_rows.append(zero_cell_false_positive_table(forecast, targets[split_name], area_weights, model_name, split_name))
zero_fp_df = pd.concat(zero_fp_rows, ignore_index=True) if zero_fp_rows else pd.DataFrame()
zero_fp_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_eval_zero_fp.csv"
zero_fp_df.to_csv(zero_fp_path, index=False)
print(f"Saved zero-cell diagnostics: {zero_fp_path}")
if not zero_fp_df.empty:
    display(zero_fp_df.sort_values(["split", "zero_forecast_mean"]))


def build_eval_summary(metrics_df: pd.DataFrame, zero_fp_df: pd.DataFrame, manifest_df: pd.DataFrame | None = None) -> pd.DataFrame:
    """One-row-per-model/split summary for final validation/test evaluation."""
    if metrics_df.empty:
        return pd.DataFrame()
    headline = metrics_df[metrics_df["subset"].isin(HEADLINE_SUBSETS)].copy()
    pieces = []
    keep_cols = [
        "model", "split", "crps", "rmse", "mae", "bias", "correlation", "obs_mean", "forecast_mean",
        "ensemble_spread_mean", "crps_skill_vs_climatology", "crps_skill_vs_persistence",
        "crps_skill_vs_deterministic_unet_mild2_tail2", "rmse_skill_vs_climatology",
        "rmse_skill_vs_persistence", "rmse_skill_vs_deterministic_unet_mild2_tail2",
    ]
    for subset in HEADLINE_SUBSETS:
        sub = headline[headline["subset"] == subset].loc[:, [c for c in keep_cols if c in headline.columns]].copy()
        sub = sub.rename(columns={c: f"{subset}__{c}" for c in sub.columns if c not in {"model", "split"}})
        pieces.append(sub)
    if not pieces:
        return pd.DataFrame()
    summary = pieces[0]
    for piece in pieces[1:]:
        summary = summary.merge(piece, on=["model", "split"], how="outer")
    z = zero_fp_df.copy() if not zero_fp_df.empty else pd.DataFrame()
    if not z.empty:
        summary = summary.merge(z, on=["model", "split"], how="left")
    if manifest_df is not None and not manifest_df.empty:
        m = manifest_df.copy()
        summary = summary.merge(m, on=["model", "split"], how="left")
    sort_cols = [c for c in ["split", "all__crps"] if c in summary.columns]
    if sort_cols:
        summary = summary.sort_values(sort_cols)
    return summary

eval_summary_df = build_eval_summary(
    metrics_df if "metrics_df" in globals() else pd.DataFrame(),
    zero_fp_df,
    validation_sampler_manifest_df if "validation_sampler_manifest_df" in globals() else pd.DataFrame(),
)
eval_summary_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_eval_summary.csv"
eval_summary_df.to_csv(eval_summary_path, index=False)
print(f"Saved final evaluation summary: {eval_summary_path}")
if not eval_summary_df.empty:
    display(eval_summary_df)


## 11A. v6 transformed-target overfit calibration

This section trains only a tiny train subset. It separates three questions before a full validation run:

1. Can the denoiser reconstruct known noised training targets at different timesteps?
2. Does raw or EMA sampling behave better on the tiny overfit set?
3. Which transformed-space `x0` clipping range avoids both saturated false fires and over-conservative flat maps?

Full validation remains disabled until this section produces a believable train-debug map.


In [ ]:
class DatasetTimeSubset(Dataset):
    """A lightweight time-indexed view used for overfit and sampler debugging."""

    def __init__(self, base: WildfireDiffusionDataset, indices: list[int] | np.ndarray, split_name: str):
        self.base = base
        self.indices = np.asarray(indices, dtype="int64")
        self.split_name = split_name
        self.channels = base.channels
        self.times = pd.DatetimeIndex(base.times[self.indices])
        self.lat = base.lat
        self.lon = base.lon

    def __len__(self) -> int:
        return int(len(self.indices))

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor | str]:
        return self.base[int(self.indices[idx])]


def select_debug_month_indices(target: xr.DataArray, n_months: int | None = OVERFIT_N_MONTHS) -> np.ndarray:
    """Pick a small, deterministic mix of high-burn and spread-out train months."""
    n_total = int(target.sizes["time"])
    if n_months is None or n_months >= n_total:
        return np.arange(n_total, dtype="int64")

    n_months = max(1, int(n_months))
    monthly_total = target.where(land_mask).fillna(0).sum(("lat", "lon")).values.astype("float64")
    all_ranked = list(np.argsort(monthly_total)[::-1])
    fire_mask = np.isin(pd.DatetimeIndex(target.time.values).month, FIRE_SEASON_MONTHS)
    fire_indices = np.where(fire_mask)[0]
    fire_ranked = list(fire_indices[np.argsort(monthly_total[fire_indices])[::-1]]) if len(fire_indices) else []
    spaced = list(np.linspace(0, n_total - 1, n_months, dtype="int64"))

    selected: list[int] = []
    for pool in [all_ranked[: max(2, n_months // 3)], fire_ranked[: max(2, n_months // 3)], spaced]:
        for idx in pool:
            idx = int(idx)
            if idx not in selected:
                selected.append(idx)
            if len(selected) >= n_months:
                break
        if len(selected) >= n_months:
            break
    return np.array(sorted(selected), dtype="int64")


def make_optimizer_and_scheduler(model_for_training: nn.Module, lr: float, num_steps: int):
    optimizer_local = torch.optim.AdamW(model_for_training.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler_local = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_local, T_max=max(num_steps, 1), eta_min=MIN_LEARNING_RATE)
    scaler_local = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda" and USE_AMP))
    return optimizer_local, scheduler_local, scaler_local


def train_overfit_debug_model(train_debug_dataset: DatasetTimeSubset) -> tuple[nn.Module, EMA, pd.DataFrame]:
    """Train a fresh model on a tiny subset to check whether the recipe can overfit."""
    overfit_model = ConditionalUNetDenoiser(
        cond_channels=cond_channels,
        base_width=BASE_WIDTH,
        time_dim=TIME_EMB_DIM,
        dropout=DROPOUT,
    ).to(DEVICE)
    overfit_ema = EMA(overfit_model, OVERFIT_EMA_DECAY)
    overfit_schedule = DiffusionSchedule(DDPM_TIMESTEPS, DEVICE)
    overfit_loader = DataLoader(
        train_debug_dataset,
        batch_size=min(OVERFIT_BATCH_SIZE, len(train_debug_dataset)),
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )
    optimizer_local, scheduler_local, scaler_local = make_optimizer_and_scheduler(
        overfit_model, OVERFIT_LEARNING_RATE, OVERFIT_NUM_STEPS
    )

    iterator = iter(overfit_loader)
    records = []
    recent_losses: list[float] = []
    pbar = tqdm(range(1, OVERFIT_NUM_STEPS + 1), desc="overfit debug")
    for step in pbar:
        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(overfit_loader)
            batch = next(iterator)

        overfit_model.train()
        optimizer_local.zero_grad(set_to_none=True)
        with autocast_context():
            loss = diffusion_epsilon_loss(overfit_model, overfit_schedule, batch)
        scaler_local.scale(loss).backward()
        if GRAD_CLIP_NORM is not None:
            scaler_local.unscale_(optimizer_local)
            nn.utils.clip_grad_norm_(overfit_model.parameters(), GRAD_CLIP_NORM)
        scaler_local.step(optimizer_local)
        scaler_local.update()
        scheduler_local.step()
        overfit_ema.update(overfit_model)

        loss_value = float(loss.detach().cpu())
        recent_losses.append(loss_value)
        if len(recent_losses) > OVERFIT_LOG_EVERY:
            recent_losses.pop(0)
        pbar.set_postfix(loss=np.mean(recent_losses), lr=optimizer_local.param_groups[0]["lr"])

        if step == 1 or step % OVERFIT_EVAL_EVERY == 0 or step == OVERFIT_NUM_STEPS:
            eval_loss = run_validation_loss(
                overfit_model,
                overfit_schedule,
                DataLoader(
                    train_debug_dataset,
                    batch_size=min(OVERFIT_BATCH_SIZE, len(train_debug_dataset)),
                    shuffle=False,
                    num_workers=NUM_WORKERS,
                    pin_memory=(DEVICE.type == "cuda"),
                ),
                repeats=1,
            )
            records.append({
                "step": step,
                "recent_train_loss": float(np.mean(recent_losses)),
                "train_debug_eval_loss": eval_loss,
                "x0_aux_loss_weight": X0_AUX_LOSS_WEIGHT,
                "x0_aux_loss_clip_mode": str(X0_AUX_LOSS_CLIP_MODE),
                "lr": optimizer_local.param_groups[0]["lr"],
            })

    torch.save(
        {
            "model_state_dict": overfit_model.state_dict(),
            "ema_state_dict": overfit_ema.state_dict(),
            "config": make_model_config(),
            "target_transform_stats": target_transform_stats,
            "selected_indices": train_debug_dataset.indices.tolist(),
            "overfit_num_steps": OVERFIT_NUM_STEPS,
        },
        OVERFIT_CHECKPOINT_PATH,
    )
    print(f"Saved overfit checkpoint: {OVERFIT_CHECKPOINT_PATH}")
    return overfit_model, overfit_ema, pd.DataFrame(records)


def forecast_distribution_table(
    forecast_ens: xr.DataArray,
    observation: xr.DataArray,
    weights: xr.DataArray,
    model_name: str,
    split_name: str,
    member_dim: str = "member",
) -> pd.DataFrame:
    """Compact distribution diagnostics for the ensemble mean and ensemble spread."""
    forecast_ens, observation = xr.align(forecast_ens, observation, join="inner")
    forecast_mean = forecast_ens.mean(member_dim, skipna=True)
    forecast_spread = forecast_ens.std(member_dim, skipna=True) if forecast_ens.sizes.get(member_dim, 1) > 1 else xr.zeros_like(observation)
    conditions = make_subset_conditions(observation)
    rows = []
    for subset_name, condition in conditions.items():
        vals = forecast_mean.where(condition).values.astype("float64")
        obs_vals = observation.where(condition).values.astype("float64")
        spread_vals = forecast_spread.where(condition).values.astype("float64")
        valid = np.isfinite(vals) & np.isfinite(obs_vals)
        if not valid.any():
            continue
        rows.append({
            "model": model_name,
            "split": split_name,
            "subset": subset_name,
            "n_cell_months": int(valid.sum()),
            "obs_mean_unweighted": float(np.nanmean(obs_vals[valid])),
            "forecast_mean_unweighted": float(np.nanmean(vals[valid])),
            "forecast_median_unweighted": float(np.nanmedian(vals[valid])),
            "forecast_p01_unweighted": float(np.nanpercentile(vals[valid], 1)),
            "forecast_p05_unweighted": float(np.nanpercentile(vals[valid], 5)),
            "forecast_p50_unweighted": float(np.nanpercentile(vals[valid], 50)),
            "forecast_p95_unweighted": float(np.nanpercentile(vals[valid], 95)),
            "forecast_p99_unweighted": float(np.nanpercentile(vals[valid], 99)),
            "spread_mean_unweighted": float(np.nanmean(spread_vals[valid])),
            "weighted_obs_mean": weighted_masked_mean(observation, weights, condition),
            "weighted_forecast_mean": weighted_masked_mean(forecast_mean, weights, condition),
            "weighted_spread_mean": weighted_masked_mean(forecast_spread, weights, condition),
        })
    return pd.DataFrame(rows)


def _finite_max(da: xr.DataArray) -> float:
    values = da.values
    return float(np.nanmax(values)) if np.isfinite(values).any() else np.nan


def plot_overfit_stability_panel(
    observation: xr.DataArray,
    forecasts_by_model: dict[str, xr.DataArray],
    timestamp: pd.Timestamp,
    save_path: Path,
) -> None:
    """Side-by-side map panel for overfit sampler variants."""
    obs = observation.sel(time=timestamp).where(land_mask)
    panels = [("observed", obs)]
    for model_name, forecast in forecasts_by_model.items():
        panels.append((model_name.replace(f"{DIFFUSION_MODEL_NAME}_overfit_", ""), forecast.sel(time=timestamp).mean("member", skipna=True).where(land_mask)))
    vmax = max([_finite_max(da) for _, da in panels if np.isfinite(_finite_max(da))] + [1e-6])
    fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.2), constrained_layout=True)
    if len(panels) == 1:
        axes = [axes]
    for ax, (title, da) in zip(axes, panels):
        im = ax.imshow(da.values, origin="lower", vmin=0.0, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(im, ax=axes, shrink=0.78, label="burned fraction")
    fig.suptitle(f"overfit stability/debug — train_debug — {timestamp.date()}")
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    print(f"Saved overfit stability map panel: {save_path}")
    plt.show()


def _weighted_np_mean(values: np.ndarray, weights: np.ndarray, valid: np.ndarray) -> float:
    valid = valid & np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan
    return float(np.sum(values[valid] * weights[valid]) / np.sum(weights[valid]))


@torch.no_grad()
def one_step_reconstruction_diagnostics(
    model_for_eval: nn.Module,
    schedule_for_eval: DiffusionSchedule,
    dataset: DatasetTimeSubset,
    observation: xr.DataArray,
    timesteps: tuple[int, ...] = ONE_STEP_RECON_TIMESTEPS,
    clip_modes: tuple[bool | str | None, ...] = ONE_STEP_RECON_CLIP_MODES,
    repeats: int = ONE_STEP_RECON_REPEATS,
) -> pd.DataFrame:
    """Diagnose denoiser x0 reconstruction from known noised targets, independent of full DDIM sampling."""
    model_for_eval.eval()
    loader = DataLoader(
        dataset,
        batch_size=min(OVERFIT_BATCH_SIZE, len(dataset)),
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )
    obs_np = np.asarray(observation.transpose("time", "lat", "lon").values, dtype="float32")
    land_np = np.asarray(land_mask.transpose("lat", "lon").values, dtype=bool)
    area_np = np.asarray(area_weights.transpose("lat", "lon").fillna(0).values, dtype="float32")
    weight_np = np.broadcast_to(area_np[None, :, :], obs_np.shape)
    land_3d = np.broadcast_to(land_np[None, :, :], obs_np.shape)
    zero_valid = land_3d & (obs_np <= BURN_THRESHOLD) & np.isfinite(obs_np)
    rows = []

    for timestep in timesteps:
        timestep = int(timestep)
        timestep = max(0, min(schedule_for_eval.num_timesteps - 1, timestep))
        for repeat in range(1, int(repeats) + 1):
            pred_x0_batches_unclipped = []
            true_x0_batches = []
            pred_y_by_mode: dict[str, list[np.ndarray]] = {str(mode): [] for mode in clip_modes}
            for batch in loader:
                conditioning = batch["conditioning"].to(DEVICE, non_blocking=True)
                x0 = batch["x0"].to(DEVICE, non_blocking=True)
                t = torch.full((x0.shape[0],), timestep, device=DEVICE, dtype=torch.long)
                noise = torch.randn_like(x0)
                x_t = schedule_for_eval.q_sample(x0, t, noise)
                pred_eps = model_for_eval(x_t, conditioning, t)
                pred_x0_unclipped = schedule_for_eval.predict_x0_from_eps(x_t, t, pred_eps)
                pred_x0_batches_unclipped.append(pred_x0_unclipped.detach().cpu().numpy()[:, 0])
                true_x0_batches.append(x0.detach().cpu().numpy()[:, 0])
                for mode in clip_modes:
                    pred_x0_mode = clip_predicted_x0_if_requested(pred_x0_unclipped, mode)
                    pred_y = diffusion_to_target_scale(pred_x0_mode).detach().cpu().numpy()[:, 0]
                    pred_y_by_mode[str(mode)].append(pred_y.astype("float32"))

            true_x0_np = np.concatenate(true_x0_batches, axis=0)
            pred_x0_np = np.concatenate(pred_x0_batches_unclipped, axis=0)
            x0_valid = land_3d & np.isfinite(true_x0_np) & np.isfinite(pred_x0_np)
            x0_rmse = math.sqrt(_weighted_np_mean((pred_x0_np - true_x0_np) ** 2, weight_np, x0_valid))
            x0_mae = _weighted_np_mean(np.abs(pred_x0_np - true_x0_np), weight_np, x0_valid)

            for mode in clip_modes:
                mode_key = str(mode)
                pred_y_np = np.concatenate(pred_y_by_mode[mode_key], axis=0)
                valid = land_3d & np.isfinite(obs_np) & np.isfinite(pred_y_np)
                mse = _weighted_np_mean((pred_y_np - obs_np) ** 2, weight_np, valid)
                mae = _weighted_np_mean(np.abs(pred_y_np - obs_np), weight_np, valid)
                bias = _weighted_np_mean(pred_y_np - obs_np, weight_np, valid)
                rows.append({
                    "timestep": timestep,
                    "repeat": repeat,
                    "clip_mode": mode_key,
                    "sqrt_alpha_bar": float(torch.sqrt(schedule_for_eval.alphas_cumprod[timestep]).detach().cpu()),
                    "sqrt_one_minus_alpha_bar": float(torch.sqrt(1.0 - schedule_for_eval.alphas_cumprod[timestep]).detach().cpu()),
                    "x0_diffusion_rmse_unclipped": x0_rmse,
                    "x0_diffusion_mae_unclipped": x0_mae,
                    "x0_pred_min_unclipped": float(np.nanmin(pred_x0_np[x0_valid])) if x0_valid.any() else np.nan,
                    "x0_pred_p01_unclipped": float(np.nanpercentile(pred_x0_np[x0_valid], 1)) if x0_valid.any() else np.nan,
                    "x0_pred_p99_unclipped": float(np.nanpercentile(pred_x0_np[x0_valid], 99)) if x0_valid.any() else np.nan,
                    "x0_pred_max_unclipped": float(np.nanmax(pred_x0_np[x0_valid])) if x0_valid.any() else np.nan,
                    "burned_fraction_rmse": math.sqrt(mse) if np.isfinite(mse) else np.nan,
                    "burned_fraction_mae": mae,
                    "burned_fraction_bias": bias,
                    "obs_mean": _weighted_np_mean(obs_np, weight_np, valid),
                    "recon_mean": _weighted_np_mean(pred_y_np, weight_np, valid),
                    "zero_cell_recon_mean": _weighted_np_mean(pred_y_np, weight_np, zero_valid),
                    "zero_cell_frac_gt_1e-4": float(np.mean(pred_y_np[zero_valid] > 1e-4)) if zero_valid.any() else np.nan,
                    "zero_cell_frac_gt_1e-3": float(np.mean(pred_y_np[zero_valid] > 1e-3)) if zero_valid.any() else np.nan,
                })
    return pd.DataFrame(rows)


def summarise_one_step_reconstruction(recon_df: pd.DataFrame) -> pd.DataFrame:
    if recon_df.empty:
        return recon_df
    group_cols = [c for c in ["weight_source", "timestep", "clip_mode"] if c in recon_df.columns]
    return (
        recon_df.groupby(group_cols, as_index=False)
        .mean(numeric_only=True)
        .sort_values(group_cols)
    )


def copy_ema_to_model_for_block(model_for_eval: nn.Module, ema_for_eval: EMA, use_ema: bool):
    """Copy EMA weights into a model for a temporary block; caller must restore."""
    if use_ema:
        ema_for_eval.store(model_for_eval)
        ema_for_eval.copy_to(model_for_eval)


def restore_model_after_ema_block(model_for_eval: nn.Module, ema_for_eval: EMA, use_ema: bool):
    if use_ema:
        ema_for_eval.restore(model_for_eval)


overfit_debug_paths = {}
if RUN_OVERFIT_DEBUG:
    overfit_dir = FORECAST_DIR / "overfit_debug_v6"
    overfit_dir.mkdir(parents=True, exist_ok=True)
    overfit_indices = select_debug_month_indices(targets["train"], OVERFIT_N_MONTHS)
    overfit_dataset = DatasetTimeSubset(train_dataset, overfit_indices, split_name="train_debug")
    overfit_target = targets["train"].isel(time=overfit_indices).assign_coords(
        split=("time", np.repeat("train_debug", len(overfit_indices)))
    )

    selected_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_overfit_debug_selected_train_months.csv"
    pd.DataFrame({
        "split": "train_debug",
        "time_index": overfit_indices,
        "time": [str(t.date()) for t in overfit_dataset.times],
        "land_burn_sum": overfit_target.where(land_mask).fillna(0).sum(("lat", "lon")).values,
    }).to_csv(selected_path, index=False)
    print(f"Saved overfit selected train months: {selected_path}")

    overfit_model, overfit_ema, overfit_history_df = train_overfit_debug_model(overfit_dataset)
    overfit_history_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_overfit_debug_training_history.csv"
    overfit_history_df.to_csv(overfit_history_path, index=False)
    display(overfit_history_df.tail())

    overfit_schedule = DiffusionSchedule(DDPM_TIMESTEPS, DEVICE)
    one_step_frames = []
    for weight_source in ONE_STEP_RECON_WEIGHT_SOURCES:
        use_ema_for_recon = str(weight_source).lower() == "ema"
        copy_ema_to_model_for_block(overfit_model, overfit_ema, use_ema_for_recon)
        try:
            recon_part = one_step_reconstruction_diagnostics(
                overfit_model,
                overfit_schedule,
                overfit_dataset,
                overfit_target,
                timesteps=ONE_STEP_RECON_TIMESTEPS,
                clip_modes=ONE_STEP_RECON_CLIP_MODES,
                repeats=ONE_STEP_RECON_REPEATS,
            )
            recon_part.insert(0, "weight_source", str(weight_source))
            one_step_frames.append(recon_part)
        finally:
            restore_model_after_ema_block(overfit_model, overfit_ema, use_ema_for_recon)

    one_step_df = pd.concat(one_step_frames, ignore_index=True) if one_step_frames else pd.DataFrame()
    one_step_summary_df = summarise_one_step_reconstruction(one_step_df)
    one_step_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_overfit_debug_one_step_reconstruction.csv"
    one_step_summary_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_overfit_debug_one_step_reconstruction_summary.csv"
    one_step_df.to_csv(one_step_path, index=False)
    one_step_summary_df.to_csv(one_step_summary_path, index=False)
    print(f"Saved one-step reconstruction diagnostics: {one_step_path}")
    print(f"Saved one-step reconstruction summary:     {one_step_summary_path}")
    display_cols = [
        "weight_source", "timestep", "clip_mode", "burned_fraction_rmse", "burned_fraction_bias",
        "recon_mean", "zero_cell_recon_mean", "zero_cell_frac_gt_1e-3",
        "x0_diffusion_rmse_unclipped", "x0_pred_p99_unclipped"
    ]
    display(one_step_summary_df.loc[:, [c for c in display_cols if c in one_step_summary_df.columns]])

    overfit_forecasts_by_model: dict[str, xr.DataArray] = {}
    overfit_generation_records = []
    for variant in OVERFIT_SAMPLER_VARIANTS:
        label = str(variant["label"])
        clip_mode = variant.get("clip_x0", False)
        use_ema_for_sampling = bool(variant.get("use_ema", False))
        model_name = f"{DIFFUSION_MODEL_NAME}_overfit_{label}"
        overfit_forecast_path = overfit_dir / f"{DIFFUSION_EXPERIMENT_ID}_overfit_{label}_ens{OVERFIT_ENSEMBLE_SIZE}_forecast_train_debug.nc"
        copy_ema_to_model_for_block(overfit_model, overfit_ema, use_ema_for_sampling)
        try:
            forecast = generate_diffusion_forecast(
                model=overfit_model,
                schedule=overfit_schedule,
                dataset=overfit_dataset,
                split_name="train_debug",
                ensemble_size=OVERFIT_ENSEMBLE_SIZE,
                ddim_steps=OVERFIT_DDIM_STEPS,
                ddim_eta=DDIM_ETA,
                clip_x0=clip_mode,
                use_ema_weights=use_ema_for_sampling,
                model_label=model_name,
                out_path=overfit_forecast_path,
                regenerate=OVERFIT_REGENERATE,
            )
        finally:
            restore_model_after_ema_block(overfit_model, overfit_ema, use_ema_for_sampling)
        overfit_forecasts_by_model[model_name] = forecast
        overfit_generation_records.append({
            "model": model_name,
            "split": "train_debug",
            "forecast_path": str(overfit_forecast_path),
            "clip_x0": str(clip_mode),
            "ddim_steps": OVERFIT_DDIM_STEPS,
            "ddim_eta": DDIM_ETA,
            "ensemble_size": OVERFIT_ENSEMBLE_SIZE,
            "uses_ema_weights": use_ema_for_sampling,
        })

    overfit_metrics_list = []
    overfit_zero_fp_list = []
    overfit_distribution_list = []
    for model_name, forecast in overfit_forecasts_by_model.items():
        overfit_metrics_list.append(evaluate_forecast(forecast, overfit_target, area_weights, model_name, "train_debug"))
        overfit_zero_fp_list.append(zero_cell_false_positive_table(forecast, overfit_target, area_weights, model_name, "train_debug"))
        overfit_distribution_list.append(forecast_distribution_table(forecast, overfit_target, area_weights, model_name, "train_debug"))

    overfit_generation_df = pd.DataFrame(overfit_generation_records)
    overfit_metrics_df = pd.concat(overfit_metrics_list, ignore_index=True) if overfit_metrics_list else pd.DataFrame()
    overfit_zero_fp_df = pd.concat(overfit_zero_fp_list, ignore_index=True) if overfit_zero_fp_list else pd.DataFrame()
    overfit_distribution_df = pd.concat(overfit_distribution_list, ignore_index=True) if overfit_distribution_list else pd.DataFrame()

    overfit_generation_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_overfit_debug_forecasts.csv"
    overfit_metrics_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_overfit_debug_metrics.csv"
    overfit_zero_fp_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_overfit_debug_zero_cell_false_positives.csv"
    overfit_distribution_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_overfit_debug_forecast_distribution.csv"
    overfit_generation_df.to_csv(overfit_generation_path, index=False)
    overfit_metrics_df.to_csv(overfit_metrics_path, index=False)
    overfit_zero_fp_df.to_csv(overfit_zero_fp_path, index=False)
    overfit_distribution_df.to_csv(overfit_distribution_path, index=False)

    month_totals = overfit_target.where(land_mask).fillna(0).sum(("lat", "lon"))
    largest_month = pd.Timestamp(month_totals.time.values[int(month_totals.argmax("time").values)])
    overfit_panel_path = FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_overfit_debug_largest_burn_panel.png"
    plot_overfit_stability_panel(overfit_target, overfit_forecasts_by_model, largest_month, overfit_panel_path)

    print("Overfit/debug headline metrics by sampler x0 clipping and weight-source variant:")
    display(
        overfit_metrics_df.query("subset in @HEADLINE_SUBSETS")
        .loc[:, ["model", "split", "subset", "n_members", "crps", "rmse", "mae", "bias", "obs_mean", "forecast_mean", "ensemble_spread_mean"]]
        .sort_values(["subset", "rmse"])
    )
    display(overfit_zero_fp_df.sort_values("zero_forecast_mean"))
    display(overfit_distribution_df.query("subset == 'all'").sort_values("forecast_mean_unweighted"))

    overfit_debug_paths = {
        "selected_train_months": selected_path,
        "training_history": overfit_history_path,
        "one_step_reconstruction": one_step_path,
        "one_step_reconstruction_summary": one_step_summary_path,
        "forecast_manifest": overfit_generation_path,
        "metrics": overfit_metrics_path,
        "zero_cell_false_positives": overfit_zero_fp_path,
        "forecast_distribution": overfit_distribution_path,
        "panel": overfit_panel_path,
        "checkpoint": OVERFIT_CHECKPOINT_PATH,
    }
else:
    print("RUN_OVERFIT_DEBUG=False: skipping transformed-target overfit stability check.")


## 11B. Legacy v7 tiny-overfit sampler calibration

This section is retained for reproducibility but is disabled by default in v9. Re-enable only if you need to re-run the tiny-overfit sampler calibration.


In [ ]:
def make_v7_sampler_variants() -> list[dict]:
    """Build a compact sampler calibration grid.

    All variants use no per-step x0 clipping. The controls are initial noise
    temperature, final-only transformed-space x0 clipping, and final-only raw
    burned-fraction capping after inverse transform.
    """
    variants: list[dict] = []
    for weight_source in V7_SOURCE_WEIGHT_SOURCES:
        use_ema = str(weight_source).lower() == "ema"
        for temp in V7_TEMPERATURES:
            temp_label = f"temp{int(round(float(temp) * 100)):03d}"
            # Baseline: no final control.
            variants.append({
                "label": f"{weight_source}_{temp_label}_finalnone",
                "use_ema": use_ema,
                "x_T_scale": float(temp),
                "clip_x0": False,
                "final_x0_clip": False,
                "final_y_cap": False,
            })
            # Final-only transformed x0 clipping.
            for final_clip in V7_FINAL_X0_CLIP_MODES:
                if final_clip is False or final_clip is None:
                    continue
                variants.append({
                    "label": f"{weight_source}_{temp_label}_finalx0_{final_clip}",
                    "use_ema": use_ema,
                    "x_T_scale": float(temp),
                    "clip_x0": False,
                    "final_x0_clip": final_clip,
                    "final_y_cap": False,
                })
            # Final-only raw burned-fraction caps.
            for final_cap in V7_FINAL_Y_CAP_MODES:
                if final_cap is False or final_cap is None:
                    continue
                variants.append({
                    "label": f"{weight_source}_{temp_label}_finaly_{final_cap}",
                    "use_ema": use_ema,
                    "x_T_scale": float(temp),
                    "clip_x0": False,
                    "final_x0_clip": False,
                    "final_y_cap": final_cap,
                })
    # Deduplicate labels defensively.
    seen = set()
    unique = []
    for variant in variants:
        if variant["label"] not in seen:
            unique.append(variant)
            seen.add(variant["label"])
    return unique


def load_v6_overfit_model_for_v7(checkpoint_path: Path = SOURCE_OVERFIT_CHECKPOINT_PATH) -> tuple[nn.Module, EMA, dict]:
    """Load the v6 overfit checkpoint as raw model + EMA object for sampler calibration."""
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Missing v6 overfit checkpoint: {checkpoint_path}\n"
            "Run the v6 notebook first, or copy the generated checkpoint into data/processed/models/diffusion_unet/."
        )
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    cfg = ckpt.get("config", config)
    ckpt_stats = ckpt.get("target_transform_stats", {})
    if ckpt_stats:
        for key in ["target_mode", "y_scale", "log_mean", "log_std"]:
            current_value = target_transform_stats.get(key)
            loaded_value = ckpt_stats.get(key)
            if loaded_value is not None and current_value is not None and str(current_value) != str(loaded_value):
                print(f"Warning: target transform stat differs for {key}: current={current_value}, checkpoint={loaded_value}")
    loaded = ConditionalUNetDenoiser(
        cond_channels=int(cfg.get("cond_channels", cond_channels)),
        base_width=int(cfg.get("base_width", BASE_WIDTH)),
        time_dim=int(cfg.get("time_emb_dim", TIME_EMB_DIM)),
        dropout=float(cfg.get("dropout", DROPOUT)),
    ).to(DEVICE)
    loaded.load_state_dict(ckpt["model_state_dict"])
    loaded_ema = EMA(loaded, OVERFIT_EMA_DECAY)
    if "ema_state_dict" in ckpt:
        loaded_ema.load_state_dict(ckpt["ema_state_dict"], DEVICE)
    loaded.eval()
    print(f"Loaded v6 overfit checkpoint: {checkpoint_path}")
    print(f"Selected indices in checkpoint: {ckpt.get('selected_indices')}")
    return loaded, loaded_ema, ckpt


def sampler_calibration_summary(
    metrics_df: pd.DataFrame,
    zero_fp_df: pd.DataFrame,
    distribution_df: pd.DataFrame,
) -> pd.DataFrame:
    """Merge the key diagnostics into one sortable table."""
    if metrics_df.empty:
        return pd.DataFrame()
    all_rows = metrics_df[(metrics_df["split"] == "train_debug") & (metrics_df["subset"] == "all")].copy()
    active_rows = metrics_df[(metrics_df["split"] == "train_debug") & (metrics_df["subset"] == "active_burned_gt_1e-6")].copy()
    tail_rows = metrics_df[(metrics_df["split"] == "train_debug") & (metrics_df["subset"] == "top_1pct_train_tail")].copy()
    cols_all = ["model", "crps", "rmse", "mae", "bias", "obs_mean", "forecast_mean", "ensemble_spread_mean"]
    summary = all_rows.loc[:, [c for c in cols_all if c in all_rows.columns]].rename(columns={
        "crps": "all_crps",
        "rmse": "all_rmse",
        "mae": "all_mae",
        "bias": "all_bias",
        "obs_mean": "all_obs_mean",
        "forecast_mean": "all_forecast_mean",
        "ensemble_spread_mean": "all_spread_mean",
    })
    if not active_rows.empty:
        summary = summary.merge(
            active_rows.loc[:, ["model", "obs_mean", "forecast_mean", "rmse", "crps"]].rename(columns={
                "obs_mean": "active_obs_mean",
                "forecast_mean": "active_forecast_mean",
                "rmse": "active_rmse",
                "crps": "active_crps",
            }),
            on="model",
            how="left",
        )
    if not tail_rows.empty:
        summary = summary.merge(
            tail_rows.loc[:, ["model", "obs_mean", "forecast_mean", "rmse", "crps"]].rename(columns={
                "obs_mean": "tail_obs_mean",
                "forecast_mean": "tail_forecast_mean",
                "rmse": "tail_rmse",
                "crps": "tail_crps",
            }),
            on="model",
            how="left",
        )
    if not zero_fp_df.empty:
        summary = summary.merge(zero_fp_df.drop(columns=["split"], errors="ignore"), on="model", how="left")
    manifest_params = v7_generation_df.loc[:, ["model", "x_T_scale", "clip_x0", "final_x0_clip", "final_y_cap", "uses_ema_weights"]]
    summary = summary.merge(manifest_params, on="model", how="left")

    # Calibration score: low all-cell error, sparse zero cells, and active/tail amplitude close to observed.
    eps = 1e-12
    summary["active_mean_ratio"] = summary["active_forecast_mean"] / (summary["active_obs_mean"] + eps)
    summary["tail_mean_ratio"] = summary["tail_forecast_mean"] / (summary["tail_obs_mean"] + eps)
    summary["log_active_ratio_abs"] = np.abs(np.log(np.clip(summary["active_mean_ratio"], eps, None)))
    summary["log_tail_ratio_abs"] = np.abs(np.log(np.clip(summary["tail_mean_ratio"], eps, None)))
    zero_penalty_col = "zero_frac_pred_gt_0.001" if "zero_frac_pred_gt_0.001" in summary.columns else "zero_frac_gt_1e-3"
    summary["zero_penalty"] = summary.get(zero_penalty_col, pd.Series(0.0, index=summary.index)).fillna(0.0)
    summary["calibration_score"] = (
        summary["all_rmse"].fillna(0.0)
        + 0.02 * summary["zero_penalty"]
        + 0.005 * summary["log_active_ratio_abs"].fillna(10.0)
        + 0.005 * summary["log_tail_ratio_abs"].fillna(10.0)
    )
    return summary.sort_values("calibration_score")


v7_debug_paths = {}
if RUN_V7_SAMPLER_CALIBRATION:
    v7_dir = FORECAST_DIR / "overfit_sampler_calibration_v7"
    v7_dir.mkdir(parents=True, exist_ok=True)
    v7_model, v7_ema, v7_ckpt = load_v6_overfit_model_for_v7(SOURCE_OVERFIT_CHECKPOINT_PATH)
    v7_schedule = DiffusionSchedule(int(v7_ckpt.get("config", {}).get("ddpm_timesteps", DDPM_TIMESTEPS)), DEVICE)
    selected_indices = np.array(v7_ckpt.get("selected_indices", select_debug_month_indices(targets["train"], OVERFIT_N_MONTHS)), dtype="int64")
    overfit_dataset = DatasetTimeSubset(train_dataset, selected_indices, split_name="train_debug")
    overfit_target = targets["train"].isel(time=selected_indices).assign_coords(
        split=("time", np.repeat("train_debug", len(selected_indices)))
    )

    selected_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_selected_train_months.csv"
    pd.DataFrame({
        "split": "train_debug",
        "time_index": selected_indices,
        "time": [str(t.date()) for t in overfit_dataset.times],
        "land_burn_sum": overfit_target.where(land_mask).fillna(0).sum(("lat", "lon")).values,
    }).to_csv(selected_path, index=False)
    print(f"Saved v7 selected train months: {selected_path}")

    v7_variants = make_v7_sampler_variants()
    print(f"Running {len(v7_variants)} v7 sampler variants.")

    v7_forecasts_by_model: dict[str, xr.DataArray] = {}
    v7_generation_records = []
    for variant in v7_variants:
        label = str(variant["label"])
        use_ema_for_sampling = bool(variant.get("use_ema", False))
        model_name = f"{DIFFUSION_MODEL_NAME}_{label}"
        out_path = v7_dir / f"{DIFFUSION_EXPERIMENT_ID}_{label}_ens{OVERFIT_ENSEMBLE_SIZE}_forecast_train_debug.nc"
        copy_ema_to_model_for_block(v7_model, v7_ema, use_ema_for_sampling)
        try:
            forecast = generate_diffusion_forecast(
                model=v7_model,
                schedule=v7_schedule,
                dataset=overfit_dataset,
                split_name="train_debug",
                ensemble_size=OVERFIT_ENSEMBLE_SIZE,
                ddim_steps=OVERFIT_DDIM_STEPS,
                ddim_eta=DDIM_ETA,
                clip_x0=variant.get("clip_x0", False),  # intentionally False for v7
                use_ema_weights=use_ema_for_sampling,
                model_label=model_name,
                out_path=out_path,
                regenerate=V7_SAMPLER_REGENERATE,
                x_T_scale=float(variant.get("x_T_scale", 1.0)),
                final_x0_clip=variant.get("final_x0_clip", False),
                final_y_cap=variant.get("final_y_cap", False),
            )
        finally:
            restore_model_after_ema_block(v7_model, v7_ema, use_ema_for_sampling)
        v7_forecasts_by_model[model_name] = forecast
        v7_generation_records.append({
            "model": model_name,
            "split": "train_debug",
            "forecast_path": str(forecast.attrs.get("saved_path", out_path)),
            "source_checkpoint": str(SOURCE_OVERFIT_CHECKPOINT_PATH),
            "clip_x0": str(variant.get("clip_x0", False)),
            "x_T_scale": float(variant.get("x_T_scale", 1.0)),
            "final_x0_clip": str(variant.get("final_x0_clip", False)),
            "final_y_cap": str(variant.get("final_y_cap", False)),
            "final_y_cap_value": target_burn_cap_value(variant.get("final_y_cap", False)),
            "ddim_steps": OVERFIT_DDIM_STEPS,
            "ddim_eta": DDIM_ETA,
            "ensemble_size": OVERFIT_ENSEMBLE_SIZE,
            "uses_ema_weights": use_ema_for_sampling,
        })

    v7_generation_df = pd.DataFrame(v7_generation_records)
    v7_metrics_df = pd.concat(
        [evaluate_forecast(forecast, overfit_target, area_weights, model_name, "train_debug") for model_name, forecast in v7_forecasts_by_model.items()],
        ignore_index=True,
    )
    v7_zero_fp_df = pd.concat(
        [zero_cell_false_positive_table(forecast, overfit_target, area_weights, model_name, "train_debug") for model_name, forecast in v7_forecasts_by_model.items()],
        ignore_index=True,
    )
    v7_distribution_df = pd.concat(
        [forecast_distribution_table(forecast, overfit_target, area_weights, model_name, "train_debug") for model_name, forecast in v7_forecasts_by_model.items()],
        ignore_index=True,
    )
    v7_summary_df = sampler_calibration_summary(v7_metrics_df, v7_zero_fp_df, v7_distribution_df)

    v7_manifest_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_forecast_manifest.csv"
    v7_metrics_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_metrics.csv"
    v7_zero_fp_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_zero_cell_false_positives.csv"
    v7_distribution_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_forecast_distribution.csv"
    v7_summary_path = METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_calibration_summary.csv"
    v7_generation_df.to_csv(v7_manifest_path, index=False)
    v7_metrics_df.to_csv(v7_metrics_path, index=False)
    v7_zero_fp_df.to_csv(v7_zero_fp_path, index=False)
    v7_distribution_df.to_csv(v7_distribution_path, index=False)
    v7_summary_df.to_csv(v7_summary_path, index=False)

    month_totals = overfit_target.where(land_mask).fillna(0).sum(("lat", "lon"))
    largest_month = pd.Timestamp(month_totals.time.values[int(month_totals.argmax("time").values)])
    # Full panel can be very wide; the compact panel includes the raw temp=1 baseline plus top-scoring variants.
    baseline_names = [m for m in v7_forecasts_by_model if m.endswith("raw_temp100_finalnone")]
    top_names = list(v7_summary_df.head(V7_PANEL_MAX_VARIANTS)["model"].values) if not v7_summary_df.empty else []
    selected_panel_names = []
    for name in baseline_names + top_names:
        if name in v7_forecasts_by_model and name not in selected_panel_names:
            selected_panel_names.append(name)
    selected_panel_forecasts = {name: v7_forecasts_by_model[name] for name in selected_panel_names}
    v7_panel_path = FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_top_sampler_variants_panel.png"
    plot_overfit_stability_panel(overfit_target, selected_panel_forecasts, largest_month, v7_panel_path)

    print("v7 sampler calibration summary — lower calibration_score is better:")
    display_cols = [
        "model", "x_T_scale", "final_x0_clip", "final_y_cap", "all_rmse", "all_crps",
        "zero_forecast_mean", "zero_frac_pred_gt_0.001", "active_obs_mean", "active_forecast_mean",
        "tail_obs_mean", "tail_forecast_mean", "active_mean_ratio", "tail_mean_ratio", "calibration_score",
    ]
    display(v7_summary_df.loc[:, [c for c in display_cols if c in v7_summary_df.columns]].head(20))
    display(v7_zero_fp_df.sort_values("zero_forecast_mean").head(20))

    v7_debug_paths = {
        "selected_train_months": selected_path,
        "forecast_manifest": v7_manifest_path,
        "metrics": v7_metrics_path,
        "zero_cell_false_positives": v7_zero_fp_path,
        "forecast_distribution": v7_distribution_path,
        "calibration_summary": v7_summary_path,
        "panel": v7_panel_path,
        "source_checkpoint": SOURCE_OVERFIT_CHECKPOINT_PATH,
    }
    overfit_debug_paths = v7_debug_paths
else:
    print("RUN_V7_SAMPLER_CALIBRATION=False: skipping v7 sampler-only calibration.")


## 11B. Legacy sampler/debug report for a full checkpoint

This section is retained for later full-checkpoint debugging. It is disabled by default in v6 because the current priority is the tiny overfit calibration above.


In [ ]:
def forecast_distribution_table(
    forecast_ens: xr.DataArray,
    observation: xr.DataArray,
    weights: xr.DataArray,
    model_name: str,
    split_name: str,
    member_dim: str = "member",
) -> pd.DataFrame:
    """Compact distribution diagnostics for the ensemble mean and ensemble spread."""
    forecast_ens, observation = xr.align(forecast_ens, observation, join="inner")
    forecast_mean = forecast_ens.mean(member_dim, skipna=True)
    forecast_spread = forecast_ens.std(member_dim, skipna=True) if forecast_ens.sizes.get(member_dim, 1) > 1 else xr.zeros_like(observation)
    conditions = make_subset_conditions(observation)
    rows = []
    for subset_name, condition in conditions.items():
        vals = forecast_mean.where(condition).values.astype("float64")
        obs_vals = observation.where(condition).values.astype("float64")
        spread_vals = forecast_spread.where(condition).values.astype("float64")
        valid = np.isfinite(vals) & np.isfinite(obs_vals)
        if not valid.any():
            continue
        rows.append({
            "model": model_name,
            "split": split_name,
            "subset": subset_name,
            "n_cell_months": int(valid.sum()),
            "obs_mean_unweighted": float(np.nanmean(obs_vals[valid])),
            "forecast_mean_unweighted": float(np.nanmean(vals[valid])),
            "forecast_median_unweighted": float(np.nanmedian(vals[valid])),
            "forecast_p01_unweighted": float(np.nanpercentile(vals[valid], 1)),
            "forecast_p05_unweighted": float(np.nanpercentile(vals[valid], 5)),
            "forecast_p50_unweighted": float(np.nanpercentile(vals[valid], 50)),
            "forecast_p95_unweighted": float(np.nanpercentile(vals[valid], 95)),
            "forecast_p99_unweighted": float(np.nanpercentile(vals[valid], 99)),
            "spread_mean_unweighted": float(np.nanmean(spread_vals[valid])),
            "weighted_obs_mean": weighted_masked_mean(observation, weights, condition),
            "weighted_forecast_mean": weighted_masked_mean(forecast_mean, weights, condition),
            "weighted_spread_mean": weighted_masked_mean(forecast_spread, weights, condition),
        })
    return pd.DataFrame(rows)


sampler_debug_generation_path = METRIC_DIR / "diffusion_sampler_debug_forecasts.csv"
sampler_debug_metrics_path = METRIC_DIR / "diffusion_sampler_debug_metrics.csv"
sampler_debug_zero_fp_path = METRIC_DIR / "diffusion_sampler_debug_zero_cell_false_positives.csv"
sampler_debug_distribution_path = METRIC_DIR / "diffusion_sampler_debug_forecast_distribution.csv"

if sampler_debug_records:
    sampler_debug_generation_df = pd.DataFrame(sampler_debug_records)
    sampler_debug_generation_df.to_csv(sampler_debug_generation_path, index=False)

    debug_metrics = []
    debug_zero_fp = []
    debug_distribution = []
    for (model_name, split_name), forecast in sampler_debug_forecasts.items():
        obs = sampler_debug_targets[split_name]
        debug_metrics.append(evaluate_forecast(forecast, obs, area_weights, model_name, split_name))
        debug_zero_fp.append(zero_cell_false_positive_table(forecast, obs, area_weights, model_name, split_name))
        debug_distribution.append(forecast_distribution_table(forecast, obs, area_weights, model_name, split_name))

    sampler_debug_metrics_df = pd.concat(debug_metrics, ignore_index=True) if debug_metrics else pd.DataFrame()
    sampler_debug_zero_fp_df = pd.concat(debug_zero_fp, ignore_index=True) if debug_zero_fp else pd.DataFrame()
    sampler_debug_distribution_df = pd.concat(debug_distribution, ignore_index=True) if debug_distribution else pd.DataFrame()

    sampler_debug_metrics_df.to_csv(sampler_debug_metrics_path, index=False)
    sampler_debug_zero_fp_df.to_csv(sampler_debug_zero_fp_path, index=False)
    sampler_debug_distribution_df.to_csv(sampler_debug_distribution_path, index=False)

    print(f"Saved sampler-debug forecast manifest: {sampler_debug_generation_path}")
    print(f"Saved sampler-debug metrics:           {sampler_debug_metrics_path}")
    print(f"Saved sampler-debug zero-cell table:   {sampler_debug_zero_fp_path}")
    print(f"Saved sampler-debug distribution:      {sampler_debug_distribution_path}")

    display(
        sampler_debug_metrics_df.query("subset in ['all', 'fire_season_Jul-Oct', 'active_burned_gt_1e-6', 'top_1pct_train_tail']")
        .sort_values(["split", "subset", "crps"])
        .loc[:, ["model", "split", "subset", "n_members", "crps", "rmse", "mae", "bias", "obs_mean", "forecast_mean", "ensemble_spread_mean"]]
    )
    display(sampler_debug_zero_fp_df.sort_values(["split", "zero_forecast_mean"]))
    display(
        sampler_debug_distribution_df.query("subset == 'all'")
        .sort_values(["split", "forecast_mean_unweighted"])
    )
else:
    print("No sampler-debug forecasts were generated in this run.")


In [ ]:
def _finite_max(da: xr.DataArray) -> float:
    values = np.asarray(da.values, dtype="float64")
    return float(np.nanmax(values)) if np.isfinite(values).any() else np.nan


def month_with_largest_burn(
    observation: xr.DataArray,
    fire_season_only: bool = False,
    year: int | None = None,
) -> pd.Timestamp:
    obs = observation.where(land_mask)
    if year is not None:
        obs = obs.sel(time=obs.time.dt.year == int(year))
    if fire_season_only:
        obs = obs.sel(time=obs.time.dt.month.isin(FIRE_SEASON_MONTHS))
    if obs.sizes.get("time", 0) == 0:
        raise ValueError(f"No months available for year={year}, fire_season_only={fire_season_only}")
    monthly_total = obs.fillna(0).sum(("lat", "lon"))
    idx = int(monthly_total.argmax("time").values)
    return pd.Timestamp(monthly_total.time.values[idx])


def plot_map_panel(
    timestamp: pd.Timestamp,
    forecasts: dict[tuple[str, str], xr.DataArray],
    split_name: str,
    save_path: Path | None = None,
    max_models: int = 4,
) -> None:
    obs = targets[split_name].sel(time=timestamp).where(land_mask)
    model_names = [name for (name, split) in forecasts.keys() if split == split_name]
    preferred_order = [
        DIFFUSION_MODEL_NAME,
        "deterministic_unet_mild2_tail2",
        "climatology",
        "persistence",
    ]
    model_names = [m for m in preferred_order if m in model_names] + [m for m in model_names if m not in preferred_order]
    model_names = model_names[:max_models]

    panels = [("observed", obs)]
    for model_name in model_names:
        forecast = forecasts[(model_name, split_name)].sel(time=timestamp).mean("member", skipna=True).where(land_mask)
        panels.append((model_name, forecast))

    vmax = max([_finite_max(da) for _, da in panels if np.isfinite(_finite_max(da))] + [1e-6])
    ncols = len(panels)
    fig, axes = plt.subplots(1, ncols, figsize=(4.0 * ncols, 4.2), constrained_layout=True)
    if ncols == 1:
        axes = [axes]
    for ax, (title, da) in zip(axes, panels):
        im = ax.imshow(da.values, origin="lower", vmin=0.0, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(im, ax=axes, shrink=0.78, label="burned fraction")
    fig.suptitle(f"{split_name} — {timestamp.date()} | final frozen ensemble means")
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        print(f"Saved map panel: {save_path}")
    plt.show()


if all_forecasts:
    # Reproducibility panels on validation.
    val_largest = month_with_largest_burn(targets["val"], fire_season_only=False)
    val_largest_fire = month_with_largest_burn(targets["val"], fire_season_only=True)
    plot_map_panel(
        val_largest,
        all_forecasts,
        split_name="val",
        save_path=FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_val_largest_burn_panel.png",
    )
    plot_map_panel(
        val_largest_fire,
        all_forecasts,
        split_name="val",
        save_path=FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_val_largest_fire_season_panel.png",
    )

    if RUN_TEST_EVAL:
        # Test/case-study panels. These are selected by observed burned area, not by model score.
        test_largest = month_with_largest_burn(targets["test"], fire_season_only=False)
        test_largest_fire = month_with_largest_burn(targets["test"], fire_season_only=True)
        portugal_2017 = month_with_largest_burn(targets["test"], fire_season_only=False, year=2017)
        iberia_2022_summer = month_with_largest_burn(targets["test"], fire_season_only=True, year=2022)

        plot_map_panel(
            test_largest,
            all_forecasts,
            split_name="test",
            save_path=FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_test_largest_burn_panel.png",
        )
        plot_map_panel(
            test_largest_fire,
            all_forecasts,
            split_name="test",
            save_path=FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_test_largest_fire_season_panel.png",
        )
        plot_map_panel(
            portugal_2017,
            all_forecasts,
            split_name="test",
            save_path=FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_test_2017_largest_burn_panel.png",
        )
        plot_map_panel(
            iberia_2022_summer,
            all_forecasts,
            split_name="test",
            save_path=FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_test_2022_largest_fire_season_panel.png",
        )


## 11C. Legacy sampler/debug map panels

These panels are only produced if `RUN_SAMPLER_DEBUG_CHECK=True`.


In [ ]:
def month_with_largest_burn_for_split(observation: xr.DataArray, fire_season_only: bool = False) -> pd.Timestamp:
    obs = observation.where(land_mask)
    if fire_season_only:
        obs = obs.sel(time=obs.time.dt.month.isin(FIRE_SEASON_MONTHS))
    monthly_total = obs.fillna(0).sum(("lat", "lon"))
    idx = int(monthly_total.argmax("time").values)
    return pd.Timestamp(monthly_total.time.values[idx])


def plot_sampler_debug_panel(
    split_name: str,
    timestamp: pd.Timestamp,
    forecasts: dict[tuple[str, str], xr.DataArray],
    save_path: Path | None = None,
) -> None:
    if split_name not in sampler_debug_targets:
        print(f"No sampler-debug target for split={split_name}")
        return
    model_names = [name for (name, split) in forecasts.keys() if split == split_name]
    if not model_names:
        print(f"No sampler-debug forecasts for split={split_name}")
        return
    model_names = sorted(model_names)
    obs = sampler_debug_targets[split_name].sel(time=timestamp).where(land_mask)
    panels = [("observed", obs)]
    for model_name in model_names:
        forecast = forecasts[(model_name, split_name)].sel(time=timestamp).mean("member", skipna=True).where(land_mask)
        panels.append((model_name, forecast))

    vmax = max([_finite_max(da) for _, da in panels if np.isfinite(_finite_max(da))] + [1e-6])
    ncols = len(panels)
    fig, axes = plt.subplots(1, ncols, figsize=(4.0 * ncols, 4.2), constrained_layout=True)
    if ncols == 1:
        axes = [axes]
    for ax, (title, da) in zip(axes, panels):
        im = ax.imshow(da.values, origin="lower", vmin=0.0, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(im, ax=axes, shrink=0.78, label="burned fraction")
    fig.suptitle(f"sampler/debug — {split_name} — {timestamp.date()}")
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        print(f"Saved sampler/debug map panel: {save_path}")
    plt.show()


if sampler_debug_forecasts:
    sampler_debug_val_panel_path = FIGURE_DIR / "diffusion_sampler_debug_val_largest_burn_panel.png"
    val_debug_month = month_with_largest_burn_for_split(sampler_debug_targets["val"], fire_season_only=False)
    plot_sampler_debug_panel("val", val_debug_month, sampler_debug_forecasts, save_path=sampler_debug_val_panel_path)

    if "train_debug" in sampler_debug_targets:
        sampler_debug_train_panel_path = FIGURE_DIR / "diffusion_sampler_debug_train_largest_burn_panel.png"
        train_debug_month = month_with_largest_burn_for_split(sampler_debug_targets["train_debug"], fire_season_only=False)
        plot_sampler_debug_panel("train_debug", train_debug_month, sampler_debug_forecasts, save_path=sampler_debug_train_panel_path)
else:
    print("No sampler/debug map panels to plot.")


## 12. Plots and lock-readiness helper

Use this after the full validation run finishes. It gives a first-pass indication of whether the diffusion model is ready to lock or whether to tune before touching the test set.

In [ ]:
def plot_training_history(history: pd.DataFrame) -> None:
    if history.empty:
        print("No training history available.")
        return
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(history["epoch"], history["train_eps_mse"], label="train eps MSE")
    ax.plot(history["epoch"], history["val_eps_mse"], label="val eps MSE")
    ax.set_xlabel("epoch")
    ax.set_ylabel("weighted epsilon MSE")
    ax.set_title("Diffusion training curve")
    ax.legend()
    plt.show()


def plot_metric_bars(metrics: pd.DataFrame, split_name: str = "val", subset_name: str = "all", metric: str = "crps") -> None:
    df = metrics[(metrics["split"] == split_name) & (metrics["subset"] == subset_name)].copy()
    if df.empty:
        print(f"No rows for split={split_name}, subset={subset_name}")
        return
    df = df.sort_values(metric)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(df["model"], df[metric])
    ax.set_ylabel(metric)
    ax.set_title(f"{split_name} {subset_name}: {metric}")
    ax.tick_params(axis="x", rotation=35)
    plt.show()


def plot_monthly_scores(monthly: pd.DataFrame, split_name: str = "val", metric: str = "crps") -> None:
    df = monthly[monthly["split"] == split_name].copy()
    if df.empty:
        print(f"No monthly rows for split={split_name}")
        return
    fig, ax = plt.subplots(figsize=(10, 5))
    for model_name, group in df.groupby("model"):
        group = group.sort_values("month")
        ax.plot(group["month"], group[metric], marker="o", label=model_name)
    ax.set_xticks(range(1, 13))
    ax.set_xlabel("calendar month")
    ax.set_ylabel(metric)
    ax.set_title(f"{split_name} monthly {metric}")
    ax.legend()
    plt.show()


def plot_reliability(rel: pd.DataFrame, split_name: str = "val", event_name: str = "top_1pct_train_tail") -> None:
    df = rel[(rel["split"] == split_name) & (rel["event"] == event_name)].copy()
    if df.empty:
        print(f"No reliability rows for split={split_name}, event={event_name}")
        return
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], linestyle="--", label="perfect reliability")
    for model_name, group in df.groupby("model"):
        group = group.sort_values("bin_mid")
        group = group[group["n_cell_months"] > 0]
        ax.plot(group["forecast_probability_mean"], group["observed_frequency"], marker="o", label=model_name)
    ax.set_xlabel("forecast event probability")
    ax.set_ylabel("observed event frequency")
    ax.set_title(f"Reliability: {split_name}, {event_name}")
    ax.legend()
    plt.show()


def selected_diffusion_validation_model(metrics: pd.DataFrame) -> str | None:
    """Return the final frozen diffusion model name if present."""
    if DIFFUSION_MODEL_NAME in set(metrics.get("model", [])):
        return DIFFUSION_MODEL_NAME
    candidates = [m for m in metrics.get("model", pd.Series(dtype=str)).unique() if str(m).startswith(DIFFUSION_MODEL_NAME)]
    return sorted(candidates)[0] if candidates else None


def summarize_lock_readiness(metrics: pd.DataFrame, zero_fp: pd.DataFrame) -> pd.DataFrame:
    """Compact validation summary for checking the frozen diffusion recipe."""
    if metrics.empty:
        print("No metrics available yet.")
        return pd.DataFrame()
    primary_model = selected_diffusion_validation_model(metrics)
    if primary_model is None:
        print("No final diffusion validation model found in metrics.")
        return pd.DataFrame()
    val = metrics[(metrics["split"] == "val") & (metrics["subset"].isin(HEADLINE_SUBSETS))].copy()
    rows = []
    for subset in HEADLINE_SUBSETS:
        group = val[val["subset"] == subset]
        if group.empty or primary_model not in set(group["model"]):
            continue
        diff = group[group["model"] == primary_model].iloc[0]
        row = {
            "subset": subset,
            "primary_diffusion_model": primary_model,
            "diffusion_crps": diff["crps"],
            "diffusion_rmse": diff["rmse"],
            "diffusion_bias": diff["bias"],
            "diffusion_spread": diff["ensemble_spread_mean"],
            "best_crps_model": group.loc[group["crps"].idxmin(), "model"],
            "best_rmse_model": group.loc[group["rmse"].idxmin(), "model"],
            "crps_skill_vs_climatology": diff.get("crps_skill_vs_climatology", np.nan),
            "crps_skill_vs_persistence": diff.get("crps_skill_vs_persistence", np.nan),
            "crps_skill_vs_deterministic": diff.get("crps_skill_vs_deterministic_unet_mild2_tail2", np.nan),
        }
        rows.append(row)
    summary = pd.DataFrame(rows)
    display(summary)

    if not summary.empty:
        problems = []
        if (summary["best_crps_model"] != primary_model).any():
            problems.append("The primary diffusion sampler is not the best CRPS model on every headline subset.")
        if (summary["crps_skill_vs_climatology"].dropna() <= 0).any():
            problems.append("The primary diffusion sampler does not beat climatology CRPS on at least one headline subset.")
        if "deterministic_unet_mild2_tail2" in set(metrics["model"]):
            if (summary["crps_skill_vs_deterministic"].dropna() <= 0).any():
                problems.append("The primary diffusion sampler does not beat the deterministic U-Net's singleton-ensemble CRPS/MAE on at least one headline subset.")
        if not zero_fp.empty and primary_model in set(zero_fp["model"]):
            diff_zero = zero_fp[(zero_fp["split"] == "val") & (zero_fp["model"] == primary_model)]
            clim_zero = zero_fp[(zero_fp["split"] == "val") & (zero_fp["model"] == "climatology")]
            if not diff_zero.empty and not clim_zero.empty:
                if float(diff_zero["zero_forecast_p99"].iloc[0]) > max(2.0 * float(clim_zero["zero_forecast_p99"].iloc[0]), 1e-4):
                    problems.append("The primary diffusion sampler has a high p99 false-positive forecast on zero-burn cells relative to climatology.")
        if problems:
            print("Tune before locking. Main reasons:")
            for problem in problems:
                print(f"- {problem}")
        else:
            print("Validation looks lockable from the headline checks. Still inspect reliability and map panels before finalizing.")
    return summary


plot_training_history(history_df if "history_df" in globals() else pd.DataFrame())
if not metrics_df.empty:
    plot_metric_bars(metrics_df, split_name="val", subset_name="all", metric="crps")
    plot_metric_bars(metrics_df, split_name="val", subset_name="fire_season_Jul-Oct", metric="crps")
    plot_monthly_scores(monthly_metrics_df, split_name="val", metric="crps")
    plot_reliability(reliability_df, split_name="val", event_name="top_1pct_train_tail")
    readiness_summary_df = summarize_lock_readiness(metrics_df, zero_fp_df)

## 13. Artefacts written by this notebook

For the v10 training-objective run, share the validation CSVs listed in the final cell after execution. Forecast NetCDFs are not needed unless the CSV summaries are inconclusive. To run the optional comparison, change `D10_OBJECTIVE_PRESET` from `"a"` to `"b"`, run the notebook again, and send the corresponding `d10b_*` files too.


In [ ]:
artefacts = [
    {"kind": "config", "path": str(CONFIG_PATH)},
    {"kind": "target_transform_stats", "path": str(TARGET_TRANSFORM_STATS_PATH)},
    {"kind": "best_checkpoint", "path": str(BEST_CHECKPOINT_PATH)},
    {"kind": "last_checkpoint", "path": str(LAST_CHECKPOINT_PATH)},
    {"kind": "training_history", "path": str(HISTORY_PATH)},
    {"kind": "validation_sampler_manifest", "path": str(VALIDATION_SAMPLER_MANIFEST_PATH)},
    {"kind": "eval_comparison_metrics", "path": str(metrics_path if "metrics_path" in globals() else METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_eval_metrics.csv")},
    {"kind": "eval_monthly_metrics", "path": str(monthly_metrics_path if "monthly_metrics_path" in globals() else METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_eval_monthly.csv")},
    {"kind": "eval_reliability", "path": str(reliability_path if "reliability_path" in globals() else METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_eval_reliability.csv")},
    {"kind": "eval_zero_cell_false_positives", "path": str(zero_fp_path if "zero_fp_path" in globals() else METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_eval_zero_fp.csv")},
    {"kind": "eval_forecast_distribution", "path": str(forecast_distribution_path if "forecast_distribution_path" in globals() else METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_eval_distribution.csv")},
    {"kind": "eval_summary", "path": str(eval_summary_path if "eval_summary_path" in globals() else METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_eval_summary.csv")},
    {"kind": "postcal_manifest", "path": str(postcal_manifest_path if "postcal_manifest_path" in globals() else METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_postcal_manifest.csv")},
    {"kind": "postcal_metrics", "path": str(postcal_metrics_path if "postcal_metrics_path" in globals() else METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_postcal_metrics.csv")},
    {"kind": "postcal_zero_fp", "path": str(postcal_zero_fp_path if "postcal_zero_fp_path" in globals() else METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_postcal_zero_fp.csv")},
    {"kind": "postcal_distribution", "path": str(postcal_distribution_path if "postcal_distribution_path" in globals() else METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_postcal_distribution.csv")},
    {"kind": "postcal_summary", "path": str(postcal_summary_path if "postcal_summary_path" in globals() else METRIC_DIR / f"{DIFFUSION_MODEL_NAME}_postcal_summary.csv")},
    {"kind": "val_largest_burn_map_panel", "path": str(FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_val_largest_burn_panel.png")},
    {"kind": "val_largest_fire_season_map_panel", "path": str(FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_val_largest_fire_season_panel.png")},
    {"kind": "test_largest_burn_map_panel", "path": str(FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_test_largest_burn_panel.png")},
    {"kind": "test_largest_fire_season_map_panel", "path": str(FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_test_largest_fire_season_panel.png")},
    {"kind": "test_2017_largest_burn_map_panel", "path": str(FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_test_2017_largest_burn_panel.png")},
    {"kind": "test_2022_largest_fire_season_map_panel", "path": str(FIGURE_DIR / f"{DIFFUSION_MODEL_NAME}_test_2022_largest_fire_season_panel.png")},
    {"kind": "legacy_source_v6_overfit_checkpoint", "path": str(SOURCE_OVERFIT_CHECKPOINT_PATH)},
]

artefacts_df = pd.DataFrame(artefacts)
display(artefacts_df)
